# ScanNet Preprocessing Pipeline

Downloads ScanNet `.sens` + `.txt` files, extracts evenly-spaced frames,
center-crops to 256x256, saves as paired `_rgb.pt`/`_depth.pt` tensors,
computes normalization stats, and packages for Google Drive upload.

**Requirements:**
- Colab with High RAM runtime (12 CPU cores, no GPU needed)
- Google Drive mounted for final upload

**Disk strategy:** Each `.sens` file is 100-200MB. We use a strict
download-extract-delete micro-loop per scene to avoid disk exhaustion.
Crash recovery via `.done` marker files.

## 1. Install & Imports

In [1]:
!pip install -q tqdm pypng

import glob
import hashlib
import json
import math
import multiprocessing
import os
import random
import re
import shutil
import struct
import subprocess
import sys
import time
import zlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

## 1b. Drive upload helper (REST API, synchronous)

Bypasses FUSE so uploads are verifiably complete on Google's servers before the function returns. Used by cells 24, 37, 55.

In [ ]:
# Synchronous Drive upload via REST API. Replaces shutil.copy2 + flush_and_unmount,
# which only flush the LOCAL FUSE cache and never wait for Google's servers.
# This function returns ONLY when Google has the complete file.

# Cached Drive service — auth runs ONCE per session, then svc is reused.
# Without this, every upload_to_drive_sync / copy_drive_file call re-runs
# auth.authenticate_user() and re-builds the service, which Colab surfaces
# as a fresh credential prompt each time.
_DRIVE_SVC = None

def _get_drive_svc():
    global _DRIVE_SVC
    if _DRIVE_SVC is None:
        from google.colab import auth
        from googleapiclient.discovery import build
        print('  Auth + connect to Drive API (one-time per session)...')
        auth.authenticate_user()
        _DRIVE_SVC = build('drive', 'v3')
    return _DRIVE_SVC


def upload_to_drive_sync(local_path, drive_folder_path, drive_filename,
                         mimetype='application/gzip', return_file_id=False):
    """Upload local_path to MyDrive/<drive_folder_path>/<drive_filename>.

    Uses Drive REST API resumable upload (handles 30+ GB).
    Returns the verified file size on Google's servers.
    Raises if upload incomplete.
    """
    from googleapiclient.http import MediaFileUpload

    svc = _get_drive_svc()

    # Drive query syntax has no parameter binding — must escape backslash and
    # apostrophe in interpolated values.
    def _q_escape(s):
        return s.replace('\\', '\\\\').replace("'", "\\'")

    # Resolve folder path -> Drive folder ID (creating subfolders as needed).
    folder_id = 'root'
    for name in drive_folder_path.strip('/').split('/'):
        if not name:
            continue
        _name_e = _q_escape(name)
        q = (f"'{folder_id}' in parents and name='{_name_e}' and "
             f"mimeType='application/vnd.google-apps.folder' and trashed=false")
        hits = svc.files().list(q=q, fields='files(id)').execute().get('files', [])
        if hits:
            folder_id = hits[0]['id']
        else:
            meta = {'name': name, 'parents': [folder_id],
                    'mimeType': 'application/vnd.google-apps.folder'}
            folder_id = svc.files().create(body=meta, fields='id').execute()['id']

    # Find existing file with this name (update vs duplicate).
    q = f"'{folder_id}' in parents and name='{_q_escape(drive_filename)}' and trashed=false"
    existing = svc.files().list(q=q, fields='files(id,size)').execute().get('files', [])

    local_size = os.path.getsize(local_path)
    print(f'  Uploading {local_size / 1024**3:.2f} GB to MyDrive/{drive_folder_path}/{drive_filename}...')

    media = MediaFileUpload(local_path, mimetype=mimetype,
                            resumable=True, chunksize=64 * 1024 * 1024)
    # Trash any extra duplicates beyond the first (consistent with copy_drive_file).
    if len(existing) > 1:
        for extra in existing[1:]:
            svc.files().update(fileId=extra['id'], body={'trashed': True}).execute()
    if existing:
        request = svc.files().update(
            fileId=existing[0]['id'], media_body=media, fields='id,size',
        )
    else:
        request = svc.files().create(
            body={'name': drive_filename, 'parents': [folder_id]},
            media_body=media, fields='id,size',
        )

    # Resumable chunked upload. next_chunk() blocks until each chunk is
    # acknowledged by Google. When response is non-None, file is complete.
    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f'    {int(status.progress() * 100)}%', end='\r', flush=True)
    print()
    drive_size = int(response['size'])
    if drive_size != local_size:
        raise RuntimeError(
            f'Drive API reports size {drive_size} != local {local_size}. '
            f'Upload incomplete despite resumable protocol — retry the cell.'
        )
    print(f'  VERIFIED on Google servers: {drive_size / 1024**3:.2f} GB')
    if return_file_id:
        return response['id']
    return drive_size


def ensure_pigz():
    """Ensure pigz is available; raise loudly if install fails. Cells 28/41/59/27
    invoke `tar -I pigz -p N`, which produces an opaque "Cannot execute pigz" error
    if the binary is missing."""
    if subprocess.run(['which', 'pigz'], capture_output=True).returncode == 0:
        return
    print('Installing pigz...')
    r = subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'pigz'],
        capture_output=True, text=True, timeout=120,
    )
    if r.returncode != 0:
        raise RuntimeError(
            f'pigz install failed (exit {r.returncode}). Without pigz the tar steps '
            f'in cells 28/41/59/27 will fail with "Cannot execute pigz". '
            f'apt stderr: {r.stderr[-500:]}'
        )
    if subprocess.run(['which', 'pigz'], capture_output=True).returncode != 0:
        raise RuntimeError('pigz install reported success but binary still missing.')


def copy_drive_file(source_file_id, drive_folder_path, new_filename):
    """Server-side copy of an existing Drive file to a new name in the same
    or another folder. No bytes cross the wire — only metadata (name + parent).
    Use after upload_to_drive_sync(..., return_file_id=True) to make a second
    Drive copy of a large tarball without re-uploading it.
    """
    svc = _get_drive_svc()

    # Same escape helper as above (defined locally so copy_drive_file is self-contained).
    def _q_escape(s):
        return s.replace('\\', '\\\\').replace("'", "\\'")

    folder_id = 'root'
    for name in drive_folder_path.strip('/').split('/'):
        if not name:
            continue
        _name_e = _q_escape(name)
        q = (f"'{folder_id}' in parents and name='{_name_e}' and "
             f"mimeType='application/vnd.google-apps.folder' and trashed=false")
        hits = svc.files().list(q=q, fields='files(id)').execute().get('files', [])
        if hits:
            folder_id = hits[0]['id']
        else:
            meta = {'name': name, 'parents': [folder_id],
                    'mimeType': 'application/vnd.google-apps.folder'}
            folder_id = svc.files().create(body=meta, fields='id').execute()['id']

    # Move any existing files with the target name to Trash (recoverable for 30 days).
    # Drive's REST delete bypasses Trash; using update({'trashed': True}) instead
    # so an accidental overwrite of the wrong file is recoverable.
    q = f"'{folder_id}' in parents and name='{_q_escape(new_filename)}' and trashed=false"
    for existing in svc.files().list(q=q, fields='files(id)').execute().get('files', []):
        svc.files().update(fileId=existing['id'], body={'trashed': True}).execute()

    print(f'  Server-side copy: {new_filename} (no bytes uploaded)')
    body = {'name': new_filename, 'parents': [folder_id]}
    result = svc.files().copy(fileId=source_file_id, body=body, fields='id,size').execute()
    print(f'  VERIFIED: {int(result["size"]) / 1024**3:.2f} GB on Google servers')
    return result['id']

## 2. Configuration

In [ ]:
BASE_OUT_DIR     = '/content/scannet_pretrain_256'
TMP_DIR          = '/content/scannet_tmp'
DRIVE            = '/content/drive/MyDrive/datasets'

FRAMES_PER_SCENE = 85
MIN_FRAMES       = 10
TARGET_SIZE      = 256
# MAX_WORKERS is the safe default for everything; use the more specific
# pools below where appropriate.
MAX_WORKERS      = min(8, os.cpu_count() or 1)   # general default
# .sens downloads from TUM rate-limit past ~8 concurrent connections.
NETWORK_WORKERS  = min(8, os.cpu_count() or 1)
# CPU/IO-bound work (validation, norm-stats threading, pigz) can use all cores.
CPU_WORKERS      = min(16, os.cpu_count() or 1)

# Blur rejection: Laplacian variance below this threshold indicates
# motion blur, which causes RGB-depth misalignment in handheld scans.
# Blurry frames teach fusion layers that cross-modal edges are unreliable.
# Set to 0 to disable blur rejection.
BLUR_THRESHOLD   = 20.0
# How many frames to search around a blurry sample before giving up
BLUR_SEARCH_WINDOW = 5

# === Pass 2 (rare-class supplement) config ===
# Pass 2 adds an additional ~PASS2_TARGET frames to scenes whose class has
# fewer than PASS2_RARE_THRESHOLD scenes. This boosts rare-class total frame
# count without violating the temporal-duplicate constraint:
#   MIN_GAP_FRAMES = 2*BLUR_SEARCH_WINDOW + 1 = 11 -> adjacent extracted frames
#   are at least 11 source-frames apart (~0.37s @ 30Hz). Pass 2 frames are
#   selected to sit AT LEAST MIN_GAP_FRAMES from any Pass 1 frame.
ENABLE_PASS2          = True
PASS2_RARE_THRESHOLD  = 50    # classes with < this many scenes get Pass 2
PASS2_TARGET          = 85    # additional frames per rare-class scene (best-effort)
MIN_GAP_FRAMES        = 11    # min source-frame distance between any two extracted frames

os.makedirs(BASE_OUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print(f'Output:  {BASE_OUT_DIR}')
print(f'Temp:    {TMP_DIR}')
print(f'Drive:   {DRIVE}')
print(f'Blur threshold: {BLUR_THRESHOLD} (0=disabled)')
print(f'Max workers count: {MAX_WORKERS}')

# === HHA depth-encoding config (additive — set HHA_MODE='none' to disable) ===
# 'none'     -> existing raw-depth flow only (default, backward compat)
# 'with-hha' -> save BOTH _depth.pt and _hha.pt per frame
# 'hha-only' -> save ONLY _hha.pt per frame (skips _depth.pt)
HHA_MODE = 'hha-only'
# float16 halves disk + IO vs float32. Precision loss:
#   - disparity (1/m): loses sub-cm at depths < 0.1 m (rare).
#   - height (m): full precision in the practical [-3, +3] range.
#   - angle (deg): loses sub-degree past ~64 deg; acceptable for HHA training.
# Use 'float32' if you need exact reproducibility of the HHA channels.
HHA_DTYPE = 'float16'   # 'float16' | 'float32'
assert HHA_MODE in ('none', 'with-hha', 'hha-only'), \
    f"Invalid HHA_MODE={HHA_MODE!r}. Must be one of 'none', 'with-hha', 'hha-only'."
assert HHA_DTYPE in ('float16', 'float32'), \
    f"Invalid HHA_DTYPE={HHA_DTYPE!r}. Must be 'float16' or 'float32'."

# Output dir + tarball naming switches based on HHA_MODE.
if HHA_MODE != 'none':
    HHA_OUT_DIR = '/content/scannet_pretrain_256_hha'
    HHA_TARBALL_NAME = 'scannet_pretrain_256_hha.tar.gz'
    BASE_OUT_DIR = HHA_OUT_DIR
    os.makedirs(BASE_OUT_DIR, exist_ok=True)
    print(f'HHA mode: {HHA_MODE} (dtype: {HHA_DTYPE})')
    print(f'HHA output dir: {HHA_OUT_DIR}')
else:
    HHA_TARBALL_NAME = 'scannet_pretrain_256.tar.gz'

# Drop list defaults — actual file is loaded after Drive mount (next section).
DROP_LIST_PATH = '/content/drive/MyDrive/datasets/scannet_drop_list.json'
SCANNET_DROP_LIST = None
SCANNET_DROP_SCENES = set()
SCANNET_AXIS_INVERTED = False


## 2b. Clone Repo (HHA Mode Only)
Required for the HHA-aware imports in cells 16 + 18 + the validation gate.
Skipped automatically when `HHA_MODE = 'none'`.


In [ ]:
# === Clone repo for HHA imports (gated on HHA_MODE != 'none') ===
if HHA_MODE != 'none':
    from pathlib import Path
    PROJECT_NAME = 'Multi-Stream-Neural-Networks'
    GITHUB_REPO = 'https://github.com/clingergab/Multi-Stream-Neural-Networks.git'
    LOCAL_REPO_PATH = f'/content/{PROJECT_NAME}'
    if Path(LOCAL_REPO_PATH).exists() and Path(f'{LOCAL_REPO_PATH}/.git').exists():
        print(f'Repo already exists: {LOCAL_REPO_PATH}')
        !git -C {LOCAL_REPO_PATH} pull --quiet
    else:
        if Path(LOCAL_REPO_PATH).exists():
            !rm -rf {LOCAL_REPO_PATH}
        !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if LOCAL_REPO_PATH not in sys.path:
        sys.path.insert(0, LOCAL_REPO_PATH)
    print(f'Repo on sys.path: {LOCAL_REPO_PATH}')


## 3. Scene Type Mapping

In [3]:
# 20 canonical scene types (no Misc). Order defines class indices 0-19.
SCANNET_SCENE_TYPES = [
    'apartment',
    'bathroom',
    'bedroom',
    'bookstore_library',
    'classroom',
    'closet',
    'computer_cluster',
    'conference_room',
    'copy_mail_room',
    'dining_room',
    'game_room',
    'gym',
    'hallway',
    'kitchen',
    'laundry_room',
    'living_room',
    'lobby',
    'office',
    'stairs',
    'storage_basement_garage',
]

# Aliases for raw sceneType strings that don't match canonical names.
# Populated after running Cell 14 (metadata scan).
# Keys: lowercase stripped raw sceneType -> Values: canonical name
SCENE_TYPE_ALIASES = {
    # Built from actual ScanNet metadata (Cell 14 output).
    # Raw sceneType strings that don't normalize via simple lowering + underscore.
    'bedroom / hotel': 'bedroom',
    'bedroom/hotel': 'bedroom',
    'living room / lounge': 'living_room',
    'living room/lounge': 'living_room',
    'bookstore / library': 'bookstore_library',
    'bookstore/library': 'bookstore_library',
    'copy/mail room': 'copy_mail_room',
    'copy / mail room': 'copy_mail_room',
    'storage/basement/garage': 'storage_basement_garage',
    'storage / basement / garage': 'storage_basement_garage',
    'computercluster': 'computer_cluster',
    'computer cluster': 'computer_cluster',
    'misc.': None,  # Explicit skip
    'misc': None,
}

print(f'{len(SCANNET_SCENE_TYPES)} scene types defined.')

20 scene types defined.


## 4. Mount Drive + Setup Tools

In [ ]:
from google.colab import drive
# Clear stale mount if runtime was partially reset
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    # Unmount first; only rm -rf if it's actually unmounted. Otherwise rm
    # would recurse into a still-mounted Drive and could delete real files
    # from the user's Google Drive account.
    _unmount = subprocess.run(
        ['fusermount', '-u', '/content/drive'],
        capture_output=True, text=True,
    )
    _still_mounted = subprocess.run(
        ['mountpoint', '-q', '/content/drive'],
    ).returncode == 0
    if not _still_mounted:
        !rm -rf /content/drive
    else:
        print('WARNING: /content/drive still appears mounted after fusermount; '
              'NOT running rm -rf to avoid deleting real Drive files.')
        print(f'  fusermount stderr: {_unmount.stderr.strip()}')
drive.mount('/content/drive')

TOOLS_DIR = '/content/scannet_tools'
os.makedirs(TOOLS_DIR, exist_ok=True)

# FIX #5: small retry helper. wget --tries handles transport-level retries
# but not 5xx, and a single failure on raw.githubusercontent.com aborts the
# entire setup. 3 attempts with backoff covers transient GitHub/TUM hiccups.
def _wget_retry(url, out_path, attempts=3):
    last_err = None
    for i in range(attempts):
        r = subprocess.run(
            ['wget', '-q', '--tries=2', '--timeout=30',
             '-O', out_path, url],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            return
        last_err = r.stderr[:300] or f'returncode={r.returncode}'
        # Remove zero-byte/partial output before retry.
        if os.path.exists(out_path):
            try: os.remove(out_path)
            except OSError: pass
        if i < attempts - 1:
            time.sleep(2 ** i)  # 1, 2 (last sleep skipped)
    raise RuntimeError(f'wget failed for {url} after {attempts} attempts: {last_err}')

# Download ScanNet download script (hosted by TUM, not on GitHub)
_wget_retry(
    'http://kaldir.vc.cit.tum.de/scannet/download-scannet.py',
    os.path.join(TOOLS_DIR, 'download-scannet.py'),
)

# Download SensorData.py for parsing .sens files
_wget_retry(
    'https://raw.githubusercontent.com/ScanNet/ScanNet/master/SensReader/python/SensorData.py',
    os.path.join(TOOLS_DIR, 'SensorData.py'),
)

# SensorData.py is Python 2 — patch for Python 3 compatibility
# Line-by-line patching to avoid silent failures from quote mismatches
_sd_path = os.path.join(TOOLS_DIR, 'SensorData.py')
with open(_sd_path, 'r') as f:
    _sd_lines = f.readlines()

_patched = 0
for _i, _line in enumerate(_sd_lines):
    _orig = _line

    # 1. Fix print statements: "print x" -> "print(x)"
    _match = re.match(r"^(\s*)print (.+)$", _line.rstrip())
    if _match and 'print(' not in _line:
        _sd_lines[_i] = f"{_match.group(1)}print({_match.group(2)})\n"

    # 2. Fix bytes/str join: struct.unpack('c') returns bytes in Py3
    if ".join(struct.unpack(" in _line and "b'" not in _line.split('join')[0]:
        _sd_lines[_i] = _sd_lines[_i].replace("''.join(", "b''.join(")

    # 3. Fix np.fromstring -> np.frombuffer
    if 'np.fromstring' in _sd_lines[_i]:
        _sd_lines[_i] = _sd_lines[_i].replace('np.fromstring', 'np.frombuffer')

    if _sd_lines[_i] != _orig:
        _patched += 1

with open(_sd_path, 'w') as f:
    f.writelines(_sd_lines)

assert _patched >= 7, f'Expected at least 7 patches, only applied {_patched}'
print(f'SensorData.py patched for Python 3 ({_patched} lines fixed)')

# Add tools dir to sys.path so SensorData can be imported
if TOOLS_DIR not in sys.path:
    sys.path.insert(0, TOOLS_DIR)

# Download official train/val split files
SPLITS_DIR = os.path.join(TOOLS_DIR, 'splits')
os.makedirs(SPLITS_DIR, exist_ok=True)

for split_file in ['scannetv2_train.txt', 'scannetv2_val.txt']:
    _wget_retry(
        f'https://raw.githubusercontent.com/ScanNet/ScanNet/master/Tasks/Benchmark/{split_file}',
        os.path.join(SPLITS_DIR, split_file),
    )

# Load split scene IDs
def load_split_ids(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

train_scene_ids = load_split_ids(os.path.join(SPLITS_DIR, 'scannetv2_train.txt'))
val_scene_ids   = load_split_ids(os.path.join(SPLITS_DIR, 'scannetv2_val.txt'))

print(f'Train scenes: {len(train_scene_ids)}')
print(f'Val scenes:   {len(val_scene_ids)}')
print(f'Total:        {len(train_scene_ids) + len(val_scene_ids)}')
import SensorData  # noqa: downloaded above, must import after sys.path setup

print(f'SensorData.py: {os.path.exists(os.path.join(TOOLS_DIR, "SensorData.py"))}')

## 3b. Load Phase 0 Drop List (HHA Mode Only)

Reads the drop list produced by `notebooks/scannet_phase0_validation.ipynb`. 
Must run after Drive is mounted.

In [ ]:
if HHA_MODE != 'none':
    if os.path.isfile(DROP_LIST_PATH):
        with open(DROP_LIST_PATH) as f:
            SCANNET_DROP_LIST = json.load(f)
        # #9: refuse drop lists not produced after convention verification.
        if not SCANNET_DROP_LIST.get('convention_verified', False):
            raise RuntimeError(
                f'Drop list at {DROP_LIST_PATH} is missing convention_verified=True. '
                f'It was produced before HHA convention validation; re-run '
                f'notebooks/scannet_phase0_validation.ipynb to refresh it.'
            )
        # FIX #2: bool('false') == True. Reject any non-bool value loudly
        # rather than silently flipping the convention.
        _axis_inv_raw = SCANNET_DROP_LIST.get('axis_alignment_inverted', False)
        if not isinstance(_axis_inv_raw, bool):
            raise RuntimeError(
                f'Drop list axis_alignment_inverted must be a JSON bool, got '
                f'{type(_axis_inv_raw).__name__}={_axis_inv_raw!r}. Re-run '
                f'phase 0 validation notebook.'
            )
        SCANNET_AXIS_INVERTED = _axis_inv_raw
        # CRITICAL #1: in HHA mode, scenes flagged with ANY drop reason get hard-dropped.
        # Previously 'missing_axisAlignment' scenes fell back to identity rotation,
        # which silently writes wrong HHA height/angle (camera-frame instead of
        # gravity-aligned-world-frame) for tilted-camera frames. Hard-drop them
        # so they never enter the dataset.
        n_missing_axis = 0
        for sid, reason in SCANNET_DROP_LIST.get('scenes', {}).items():
            SCANNET_DROP_SCENES.add(sid)
            if isinstance(reason, str) and reason.startswith('missing_axisAlignment'):
                n_missing_axis += 1
        if n_missing_axis > 0:
            print(f'  Note: {n_missing_axis} scenes flagged missing_axisAlignment '
                  f'are now HARD-DROPPED in HHA mode (was: identity-fallback, which '
                  f'produced silently-wrong HHA for tilted cameras).')
        print(f'Drop list: {len(SCANNET_DROP_SCENES)} scenes will be skipped, '
              f'axis_alignment_inverted={SCANNET_AXIS_INVERTED}')
    else:
        # Distinguish "Drive not mounted" from "drop list never produced".
        if not os.path.ismount('/content/drive') and not os.path.isdir('/content/drive/MyDrive'):
            raise FileNotFoundError(
                f'Drive does not appear mounted at /content/drive. '
                f'Re-run cell 12 (Mount Drive + Setup Tools).'
            )
        raise FileNotFoundError(
            f'HHA mode requires {DROP_LIST_PATH}. '
            f'Drive IS mounted but the drop-list file is missing — '
            f'run notebooks/scannet_phase0_validation.ipynb to produce it.'
        )
else:
    print('HHA_MODE=none -> skipping drop-list load.')

## 4c. Smart Restore from Drive (try first; skip extraction if available)

If a complete tarball exists on Drive, extract it locally and set `SKIP_EXTRACT=True` so the heavy extraction cells (16, 20, 24, 26, 28) bail out. Otherwise falls back to full extraction. Sets `DATA_AVAILABLE`, `RESTORE_SOURCE`, `CHANGES_MADE` flags consumed by downstream cells.

In [ ]:
# === SMART RESTORE FROM DRIVE ===
# Tries the canonical tarball first (downstream notebooks depend on it),
# then snapshot. Validates what got extracted: zero-byte files trigger
# recovery later; empty extraction triggers full re-extract.
#
# Sets four flags consumed by downstream cells:
#   DATA_AVAILABLE   = True if local tree has any usable data
#   RESTORE_SOURCE   = 'canonical' | 'snapshot' | 'fresh' | None
#   CHANGES_MADE     = True if any later cell will modify files (init False)
#   EXPECTED_TRAIN   = expected train sample count (~97k)

EXPECTED_TRAIN_MIN = 70000   # below this we treat the restore as failed
DATA_AVAILABLE = False
RESTORE_SOURCE = None
CHANGES_MADE   = False
EXPECTED_TRAIN = 97720       # post-rebalance target

train_dir = os.path.join(BASE_OUT_DIR, 'train')

# 1) If train dir already populated locally, use it as-is (kernel restart case).
if os.path.isdir(train_dir):
    n_local = sum(1 for _ in glob.glob(os.path.join(train_dir, '*', '*_rgb.pt')))
    # R6 #1: in HHA mode also count HHA files; require BOTH above threshold.
    # Was: only RGB checked, so a partial-HHA local copy passed validation
    # and downstream norm-stats/q99 would fail late.
    n_local_hha = (
        sum(1 for _ in glob.glob(os.path.join(train_dir, '*', '*_hha.pt')))
        if HHA_MODE != 'none' else n_local
    )
    if n_local >= EXPECTED_TRAIN_MIN and n_local_hha >= EXPECTED_TRAIN_MIN:
        DATA_AVAILABLE = True
        RESTORE_SOURCE = 'local'
        print(f'Local data found at {BASE_OUT_DIR}: {n_local} train RGB files'
              + (f', {n_local_hha} train HHA files.' if HHA_MODE != 'none' else '.')
              + ' Using it.')

# 2) Otherwise try Drive tarballs (canonical first, then snapshot).
if not DATA_AVAILABLE:
    # CRIT-R4 #2: use HHA_TARBALL_NAME (set in cell 6) so raw-depth mode
    # (HHA_MODE='none' -> 'scannet_pretrain_256.tar.gz') restores correctly.
    # Hardcoded HHA name made smart restore always miss in non-HHA mode.
    canonical = os.path.join(DRIVE, HHA_TARBALL_NAME)
    snapshot  = os.path.join(DRIVE, HHA_TARBALL_NAME.replace('.tar.gz', '.snapshot.tar.gz'))
    candidates = [(canonical, 'canonical'), (snapshot, 'snapshot')]
    candidates = [(p, label) for p, label in candidates if os.path.exists(p)]

    if not candidates:
        print(f'No Drive tarball found at {canonical} or {snapshot}.')
        print('Will fall back to full re-extract (cells 16-24 will run).')
    else:
        # Sort by file size; bigger usually means more complete.
        # HIGH-R4 #5: prefer canonical (downstream consumers reference the
        # canonical name) -- do NOT sort by size. The original code's
        # size-based sort silently picked snapshot when it grew larger,
        # breaking the "canonical is the source of truth" contract.
        # candidates list order is already (canonical, snapshot).

        ensure_pigz()

        for tar_path, source_label in candidates:
            size_gb = os.path.getsize(tar_path) / 1024**3
            print(f'\nTrying {source_label} tarball: {tar_path} ({size_gb:.2f} GB)')
            shutil.rmtree(BASE_OUT_DIR, ignore_errors=True)
            os.makedirs(BASE_OUT_DIR, exist_ok=True)
            result = subprocess.run(
                ['tar', '-I', f'pigz -p {CPU_WORKERS}', '-xf', tar_path,
                 '-C', os.path.dirname(BASE_OUT_DIR)],
                capture_output=True, text=True,
            )
            if result.returncode != 0:
                print(f'  tar exited {result.returncode}; stderr: {result.stderr[-300:]}')
                print(f'  Continuing with whatever was extracted.')

            # Validate extraction
            n_train_rgb = sum(1 for _ in glob.glob(os.path.join(train_dir, '*', '*_rgb.pt')))
            n_train_hha = sum(1 for _ in glob.glob(os.path.join(train_dir, '*', '*_hha.pt')))
            n_zero_rgb  = sum(1 for f in glob.glob(os.path.join(train_dir, '*', '*_rgb.pt'))
                              if os.path.getsize(f) == 0)
            n_zero_hha  = sum(1 for f in glob.glob(os.path.join(train_dir, '*', '*_hha.pt'))
                              if os.path.getsize(f) == 0)
            has_norm_stats = (
                os.path.exists(os.path.join(BASE_OUT_DIR, 'norm_stats.json'))
                and os.path.getsize(os.path.join(BASE_OUT_DIR, 'norm_stats.json')) > 0
            )
            has_class_names = (
                os.path.exists(os.path.join(BASE_OUT_DIR, 'class_names.txt'))
                and os.path.getsize(os.path.join(BASE_OUT_DIR, 'class_names.txt')) > 0
            )

            print(f'  Train RGB: {n_train_rgb}  HHA: {n_train_hha}')
            print(f'  Zero-byte RGB: {n_zero_rgb}  zero-byte HHA: {n_zero_hha}')
            print(f'  norm_stats.json: {"OK" if has_norm_stats else "MISSING"}')
            print(f'  class_names.txt: {"OK" if has_class_names else "MISSING"}')

            if (HHA_MODE == 'none' and n_train_rgb >= EXPECTED_TRAIN_MIN) or (HHA_MODE != 'none' and n_train_rgb >= EXPECTED_TRAIN_MIN and n_train_hha >= EXPECTED_TRAIN_MIN):  # R5 #4: HHA mode requires BOTH
                # Usable. If zero-byte files exist, recovery (cell 39) will fix them.
                DATA_AVAILABLE = True
                RESTORE_SOURCE = source_label
                if n_zero_rgb > 0 or n_zero_hha > 0:
                    print(f'  Restore is PARTIAL ({n_zero_rgb + n_zero_hha} zero-byte files). '
                          f'Cells 37/39 will repair.')
                else:
                    print(f'  Restore is CLEAN.')
                break  # done; do not try other tarballs
            else:
                print(f'  Restore yielded too few files. Trying next candidate (if any).')

        if not DATA_AVAILABLE:
            print('\nAll Drive tarballs unusable. Will fall back to full re-extract.')
            shutil.rmtree(BASE_OUT_DIR, ignore_errors=True)
            os.makedirs(BASE_OUT_DIR, exist_ok=True)

if DATA_AVAILABLE:
    # Pass-2-aware skip: even when data is restored, if existing markers
    # don't show Pass 2 was done, we still need to run cell 26 to add the
    # supplement. Sample a handful of .done markers to decide.
    _needs_pass2_topup = False
    if 'ENABLE_PASS2' in dir() and ENABLE_PASS2:
        import glob as _glob
        _markers = _glob.glob(os.path.join(BASE_OUT_DIR, '*.done'))
        # BUG C FIX: scan ALL markers, not just first 20. A partial earlier
        # run might have completed Pass 2 on the first N markers but not the
        # rest -- sampling could falsely conclude "all done".
        for _mf in _markers:
            try:
                with open(_mf) as _f:
                    _content = _f.read()
                if 'pass2_attempted' not in _content and 'pass2_done' not in _content:
                    _needs_pass2_topup = True
                    break
            except OSError:
                pass

    if _needs_pass2_topup:
        print(f'\n=== DATA AVAILABLE (source={RESTORE_SOURCE}) BUT Pass 2 supplement needed. ===')
        print(f'    Cells 18, 22, 26 will run incrementally — Pass 1 frames preserved,')
        print(f'    Pass 2 added to rare-class scenes only.')
        SKIP_EXTRACT = False
    else:
        print(f'\n=== DATA AVAILABLE (source={RESTORE_SOURCE}) and Pass 2 already done '
              f'(or disabled). Cells 18, 22, 26, 28, 30 will be skipped. ===')
        SKIP_EXTRACT = True
else:
    print(f'\n=== NO DATA AVAILABLE — full extraction required. ===')
    SKIP_EXTRACT = False
    RESTORE_SOURCE = 'fresh'

## 5. Download .txt Metadata (Lightweight)

Batch-download all `.txt` metadata files. These are tiny (a few KB each)
and safe to batch.

In [ ]:
if 'SKIP_EXTRACT' in dir() and SKIP_EXTRACT:
    print('SKIPPED: .txt metadata download (data restored from Drive in cell 31)')
else:
    TXT_DIR = os.path.join(TMP_DIR, 'txt_metadata')
    os.makedirs(TXT_DIR, exist_ok=True)

    all_scene_ids = train_scene_ids + val_scene_ids
    print(f'Downloading .txt metadata for {len(all_scene_ids)} scenes (32 threads)...')

    # HIGH #7: was 32 but cell 6 caps NETWORK_WORKERS=8 because TUM throttles
    # concurrent connections. Use the same cap to avoid spurious 429s on cold start.
    TXT_DL_WORKERS = NETWORK_WORKERS

    def _dl_one_txt(scene_id):
        txt_out = os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt')
        if os.path.exists(txt_out):
            return ('skip', scene_id, '')
        cmd = ['python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
               '-o', TXT_DIR, '--id', scene_id, '--type', '.txt']
        # Catch ALL exceptions (TimeoutExpired, FileNotFoundError on python3,
        # OSError on disk-full, etc.) so worker exceptions don't crash the
        # main loop via fut.result().
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=60, input='y\ny\n')
        except Exception as _e:
            return ('fail', scene_id, f'{type(_e).__name__}: {str(_e)[:200]}')
        if r.returncode != 0:
            return ('fail', scene_id, r.stderr[:200])
        return ('ok', scene_id, '')

    counts = Counter()
    failures = []
    with ThreadPoolExecutor(max_workers=TXT_DL_WORKERS) as ex:
        futures = [ex.submit(_dl_one_txt, sid) for sid in all_scene_ids]
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Downloading .txt'):
            try:
                status, sid, err = fut.result()
            except Exception as _e:
                # Worker raised before returning (e.g., uncaught earlier).
                counts['fail'] += 1
                failures.append(('<unknown>', f'{type(_e).__name__}: {str(_e)[:200]}'))
                continue
            counts[status] += 1
            if status == 'fail':
                failures.append((sid, err))

    print(f"Downloaded: {counts['ok']}, Skipped (existing): {counts['skip']}, Failed: {counts['fail']}")
    for sid, err in failures[:10]:
        print(f'  WARN {sid}: {err}')

## 6. Parse Scene Metadata

In [ ]:
def parse_scene_metadata(txt_path):
    """Parse <scene_id>.txt metadata file.

    Format: key = value (split on first '=', strip whitespace).
    Returns dict of field -> value. Key field: 'sceneType'.
    Warns on duplicate keys (last-write-wins) so silent label changes are visible.
    """
    metadata = {}
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if '=' not in line:
                continue
            key, value = line.split('=', 1)
            k, v = key.strip(), value.strip()
            if k in metadata and metadata[k] != v:
                print(f'  WARNING: duplicate key {k!r} in {txt_path}: '
                      f'{metadata[k]!r} -> {v!r} (using latest)')
            metadata[k] = v
    return metadata


def normalize_scene_type(scene_type_str):
    """Normalize raw sceneType string to canonical class name.

    Lowercases, strips whitespace, replaces spaces with underscores,
    and checks SCENE_TYPE_ALIASES. Returns None if the type is Misc
    or unrecognized (scene should be skipped).
    """
    import re as _re
    # Canonicalize whitespace: trim, collapse runs of whitespace, normalize
    # ' / ' patterns. Otherwise 'bedroom  /  hotel' (double spaces) misses
    # both the 'bedroom / hotel' AND 'bedroom/hotel' alias entries.
    raw = scene_type_str.strip().lower()
    raw = _re.sub(r'\s*/\s*', ' / ', raw)
    raw = _re.sub(r'\s+', ' ', raw)

    # Check aliases first
    if raw in SCENE_TYPE_ALIASES:
        return SCENE_TYPE_ALIASES[raw]

    # Try direct match with underscores
    canonical = raw.replace(' ', '_')
    if canonical in SCANNET_SCENE_TYPES:
        return canonical

    # Try slash variations: "bedroom / hotel" -> "bedroom"
    # Also "living room / lounge" -> "living_room"
    if '/' in raw:
        parts = [p.strip().replace(' ', '_') for p in raw.split('/')]
        for part in parts:
            if part in SCANNET_SCENE_TYPES:
                # Surface ambiguous mappings — adding to SCENE_TYPE_ALIASES
                # is preferred over relying on this fallback.
                if not hasattr(normalize_scene_type, '_logged_fallback'):
                    normalize_scene_type._logged_fallback = set()
                if raw not in normalize_scene_type._logged_fallback:
                    normalize_scene_type._logged_fallback.add(raw)
                    print(f'  [normalize_scene_type] slash-fallback: '
                          f'{scene_type_str!r} -> {part!r} '
                          f'(consider adding to SCENE_TYPE_ALIASES)')
                return part

    return None


# Quick test
print('normalize_scene_type tests:')
for test in ['Bathroom', 'bedroom / hotel', 'Living room / Lounge', 'Misc', 'Office']:
    print(f'  {test!r:30s} -> {normalize_scene_type(test)}')

## 7. Pre-Processing Metadata Scan

Parse ALL downloaded `.txt` files. Print every unique raw `sceneType`
string with counts. Identify which map to the 20 types, which need
aliases, which will be skipped.

**Review the output below and update `SCENE_TYPE_ALIASES` in Cell 6
before proceeding.**

In [ ]:
if 'SKIP_EXTRACT' in dir() and SKIP_EXTRACT:
    print('SKIPPED: metadata pre-scan (data restored from Drive in cell 31)')
else:
    raw_type_counts = Counter()
    mapped_counts = Counter()
    unmapped_types = Counter()
    scenes_with_metadata = {}

    for scene_id in all_scene_ids:
        # download-scannet.py saves to TXT_DIR/scans/<scene_id>/<scene_id>.txt
        txt_candidates = [
            os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt'),
            os.path.join(TXT_DIR, scene_id, f'{scene_id}.txt'),
            os.path.join(TXT_DIR, f'{scene_id}.txt'),
        ]
        txt_path = None
        for candidate in txt_candidates:
            if os.path.exists(candidate):
                txt_path = candidate
                break
        if txt_path is None:
            print(f'WARNING: No .txt metadata for {scene_id}')
            continue

        meta = parse_scene_metadata(txt_path)
        raw_type = meta.get('sceneType', '')
        raw_type_counts[raw_type] += 1

        canonical = normalize_scene_type(raw_type)
        if canonical:
            mapped_counts[canonical] += 1
            scenes_with_metadata[scene_id] = (canonical, txt_path)
        else:
            unmapped_types[raw_type] += 1

    print('=== All unique raw sceneType strings ===')
    for raw_type, count in sorted(raw_type_counts.items(), key=lambda x: -x[1]):
        canonical = normalize_scene_type(raw_type)
        status = f'-> {canonical}' if canonical else '** SKIPPED **'
        print(f'  {count:4d}  {raw_type!r:40s} {status}')

    print(f'\n=== Summary ===')
    print(f'Total scenes in splits:    {len(all_scene_ids)}')
    print(f'Successfully mapped:       {len(scenes_with_metadata)}')
    print(f'Will be skipped:           {sum(unmapped_types.values())}')
    print(f'Expected output frames:    ~{len(scenes_with_metadata) * FRAMES_PER_SCENE:,}')

    if unmapped_types:
        print(f'\n=== Unmapped types (need aliases or will be skipped) ===')
        for raw_type, count in sorted(unmapped_types.items(), key=lambda x: -x[1]):
            print(f'  {count:4d}  {raw_type!r}')

    print(f'\n=== Mapped class distribution ===')
    for cls, count in sorted(mapped_counts.items()):
        print(f'  {count:4d}  {cls}')

## 8. Frame Extraction Helpers

In [ ]:
import numpy as np
# CRIT #4: HHA helpers only imported when HHA mode is active. Cell 8 only
# clones the repo + adds it to sys.path when HHA_MODE != 'none'; importing
# unconditionally crashes raw-depth-mode runs with ModuleNotFoundError.
if HHA_MODE != 'none':
    from src.data_utils.hha import compute_hha
    from src.data_utils.hha.scannet_intrinsics import _orthogonalize

# Per-scene HHA metadata cache (populated by process_scene right before
# calling extract_frames_from_sens; read inside the loop above).
_HHA_SCENE_META_CACHE = {}

def laplacian_variance(rgb_frame):
    """Compute variance of Laplacian as a sharpness measure.

    Low values indicate motion blur. ScanNet is handheld video, so frames
    captured during fast camera motion have RGB-depth misalignment that
    harms fusion layer pretraining.

    Uses OpenCV's optimized C++ Laplacian instead of scipy.signal.convolve2d
    for ~50-100x speedup on the 1296x968 raw frames.

    Args:
        rgb_frame: HxWx3 uint8 numpy array.

    Returns:
        float: Variance of the Laplacian (higher = sharper).
    """
    gray = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def extract_frames_from_sens(sens_path, output_dir, scene_id,
                             num_frames=85, target_size=256,
                             blur_threshold=20.0, blur_search_window=5,
                             sd_obj=None,
                             target_indices=None,
                             seed_used_indices=None,
                             min_gap=1):
    """Extract evenly-spaced frames from a .sens file with blur rejection.

    For each selected frame index, computes the Laplacian variance (sharpness).
    If below blur_threshold, searches nearby frames (within blur_search_window)
    for a sharp replacement. This preserves spatial diversity while avoiding
    motion-blurred frames that cause RGB-depth misalignment.

    Asserts depth_shift == 1000.0 (sanity check: raw uint16 values are mm).
    Saves RGB as uint8 [3, H, W] and depth as uint16 [1, H, W].

    Returns:
        tuple: (frames_extracted, blur_replaced, blur_dropped, pose_dropped,
                decode_failed, used_indices) where
            blur_replaced = blurry frames that found a sharp neighbor,
            blur_dropped  = blurry frames with no sharp neighbor (sample lost),
            pose_dropped  = HHA-mode candidates with invalid pose,
            decode_failed = frames that failed JPEG/zlib decompress,
            used_indices  = set of int indices actually saved (for Pass 2 seeding).

    Raises:
        RuntimeError: On corrupt/unreadable .sens files or too few frames.
    """

    # HHA per-scene cache; populated by process_scene before calling.
    scene_meta = _HHA_SCENE_META_CACHE.get(scene_id)

    # FIX #5: in HHA mode, if scene_meta couldn't be loaded (axisAlignment
    # missing or .sens header parse failed), every candidate would break out
    # of the inner loop with no output. Skip the entire 85-target-iteration
    # walk + .sens parse and bail immediately. process_scene treats this as
    # a permanent failure and writes a .failed marker so re-runs skip too.
    if HHA_MODE != 'none' and scene_meta is None:
        # Must match the 6-tuple shape of the normal return path
        # (callers in cell 26 unpack 6 values).
        return 0, 0, 0, 0, 0, set()

    if sd_obj is not None:
        # Reuse the SensorData object that process_scene already loaded —
        # avoids a second ~150 MB read + frame-list walk per scene.
        sd = sd_obj
    else:
        try:
            sd = SensorData.SensorData(sens_path)
        except Exception as e:
            raise RuntimeError(f'Failed to parse {sens_path}: {e}') from e

    # Sanity check: depth_shift should be 1000.0 (raw values are mm).
    # Phase 0 verified all ScanNet scenes use 1000.0 — fail loud on anomaly
    # so the affected scene is not silently saved with wrong scale.
    if not hasattr(sd, 'depth_shift'):
        raise RuntimeError(f'{scene_id}: SensorData has no depth_shift attribute')
    # Defend against unusual capture configs that would make the bytes
    # interpreted by np.frombuffer(..., dtype=uint16) produce garbage.
    if getattr(sd, 'depth_compression_type', None) != 'zlib_ushort':
        raise RuntimeError(
            f'{scene_id}: unexpected depth_compression_type='
            f'{getattr(sd, "depth_compression_type", "MISSING")!r}; '
            f'pipeline assumes zlib_ushort'
        )
    if sd.depth_shift != 1000.0:
        raise RuntimeError(
            f'{scene_id}: unexpected depth_shift={sd.depth_shift} (expected 1000.0); '
            f'phase 0 verified all ScanNet scenes use 1000.0'
        )

    total_frames = len(sd.frames)
    if total_frames < MIN_FRAMES:
        raise RuntimeError(
            f'{scene_id}: only {total_frames} frames (min={MIN_FRAMES}), skipping'
        )

    # Select frame indices. Default: evenly-spaced via linspace (Pass 1).
    # Pass 2 caller supplies target_indices computed from Pass 1's used_indices
    # so the supplement frames sit at min_gap from any Pass 1 frame.
    if target_indices is not None:
        indices = np.asarray(target_indices, dtype=int)
        indices = indices[(indices >= 0) & (indices < total_frames)]
    else:
        n_sample = min(num_frames, total_frames)
        indices = np.linspace(0, total_frames - 1, n_sample, dtype=int)

    # NOTE: process_scene (cell 24) ensures scene_out_dir exists before
    # invoking us. Removed the per-call os.makedirs.
    extracted = 0
    blur_replaced = 0
    blur_dropped = 0
    pose_dropped = 0  # all candidates rejected by HHA pose-validity (HHA mode only)
    decode_failed = 0  # decompress_color or decompress_depth raised
    # Seed used_indices from Pass 1 if running Pass 2 — prevents re-saving Pass 1 frames
    # AND lets the min_gap check below enforce temporal distance from Pass 1 selections.
    used_indices = set(seed_used_indices) if seed_used_indices else set()
    duplicates_skipped = 0  # blur windows overlapped to the same idx

    for target_idx in indices:
        # Try the target frame first, then search nearby if blurry
        # Alternating +/- candidates with a deterministic but unbiased order:
        # for each |offset|, randomly pick which sign goes first (seeded by
        # target_idx so the sequence is reproducible).
        candidates = [target_idx]
        if blur_threshold > 0:
            _bias_rng = np.random.default_rng(target_idx)
            for offset in range(1, blur_search_window + 1):
                pos_first = bool(_bias_rng.integers(0, 2))
                signs = (1, -1) if pos_first else (-1, 1)
                for sign in signs:
                    cand = target_idx + sign * offset
                    if 0 <= cand < total_frames:
                        candidates.append(cand)

        frame_saved = False
        target_was_blurry = False
        target_pose_failed = False
        for idx in candidates:
            # Skip if a previous target_idx already saved this idx (overlapping
            # blur-search windows). Prevents double-save and inflated extracted
            # counter that would falsely pass the MIN_FRAMES check.
            if idx in used_indices:
                duplicates_skipped += 1
                continue
            # Enforce minimum temporal distance from any already-used index.
            # For Pass 1 (min_gap=1, used_indices starts empty) this only blocks
            # exact re-use, which the line above already handles. For Pass 2
            # (min_gap=11, seeded with Pass 1 indices) this guarantees no
            # extracted frame sits within ±10 frames of a Pass 1 frame.
            if min_gap > 1 and any(abs(idx - u) < min_gap for u in used_indices):
                duplicates_skipped += 1
                continue
            frame = sd.frames[idx]

            # Decompress color (JPEG) -> RGB HxWx3 uint8
            try:
                color_img = frame.decompress_color(sd.color_compression_type)
            except Exception:
                decode_failed += 1
                continue
            if color_img is None:
                decode_failed += 1
                continue

            # Blur rejection check (on full-res RGB for accuracy)
            if blur_threshold > 0:
                sharpness = laplacian_variance(color_img)
                if sharpness < blur_threshold:
                    if idx == target_idx:
                        target_was_blurry = True
                    continue  # Try next candidate

            # Decompress depth -> raw bytes -> HxW uint16
            try:
                depth_data = frame.decompress_depth(sd.depth_compression_type)
                depth_img = np.frombuffer(depth_data, dtype=np.uint16).reshape(
                    sd.depth_height, sd.depth_width
                )
            except Exception:
                decode_failed += 1
                continue

            # SPATIAL ALIGNMENT + RESIZE TO TARGET SIZE
            # SensorData.py returns raw unaligned images -- RGB at 1296x968 and
            # depth at 640x480 with different intrinsics and a ~3.8cm physical
            # offset (ScanNet GitHub Issues #28, #69, #101).
            #
            # Step 1: Resize RGB to depth resolution (640x480) for FOV alignment
            # Step 2: Resize both to target_size x target_size (256x256) to
            #         preserve full spatial coverage for scene classification
            depth_h, depth_w = depth_img.shape[:2]
            if color_img.shape[:2] != (depth_h, depth_w):
                color_img = cv2.resize(
                    color_img, (depth_w, depth_h),
                    interpolation=cv2.INTER_AREA,
                )

            # Resize both to target size (preserves full scene context)
            color_crop = cv2.resize(
                color_img, (target_size, target_size),
                interpolation=cv2.INTER_AREA,
            )
            depth_crop = cv2.resize(
                depth_img, (target_size, target_size),
                interpolation=cv2.INTER_NEAREST,  # nearest for depth to avoid interpolation artifacts
            )

            # === Pre-compute HHA BEFORE saving anything (atomicity) ===
            # If HHA is required and the pose is invalid, skip this candidate
            # and try the next one — exactly like a blur failure. This
            # guarantees every saved _rgb.pt has a paired _hha.pt and avoids
            # creating orphan files that break dataset _discover_samples().
            #
            # HHA computed at NATIVE depth resolution then nearest-resized
            # to target_size. Matches scripts/preprocess_sunrgbd_19.py.
            hha_tensor = None
            if HHA_MODE != 'none':
                if scene_meta is None:
                    # Scene meta couldn't be loaded -> can't make a paired
                    # HHA frame for any candidate. Bail on this target_idx.
                    break
                pose_4x4 = np.asarray(frame.camera_to_world, dtype=np.float64)
                # Some SensorData parser variants return a flat (16,) array.
                # Reshape if so; warn loudly if it's any other shape (would
                # otherwise silently fail every candidate -> 0 frames per scene).
                if pose_4x4.shape == (16,):
                    pose_4x4 = pose_4x4.reshape(4, 4)
                if pose_4x4.shape != (4, 4):
                    if not getattr(extract_frames_from_sens, '_warned_pose_shape', False):
                        print(f'WARNING: {scene_id} idx={idx}: pose has shape '
                              f'{pose_4x4.shape}, expected (4,4) or (16,). '
                              f'All candidates will fail HHA pose check; '
                              f'check SensorData.py patch.')
                        extract_frames_from_sens._warned_pose_shape = True
                    if idx == target_idx:
                        target_pose_failed = True
                    continue
                if not np.all(np.isfinite(pose_4x4)):
                    if idx == target_idx:
                        target_pose_failed = True
                    continue  # bad pose -> try next candidate
                K_native = scene_meta['K']
                depth_m_native = depth_img.astype(np.float32) / scene_meta['depth_shift']
                R_scannet = scene_meta['axis_alignment_rot'] @ pose_4x4[:3, :3]
                hha_native = compute_hha(
                    depth_m_native, K_native, R_scannet,
                    apply_sunrgbd_basis_swap=False,
                )  # [3, depth_h, depth_w] float32 with NaN at invalid pixels
                # Nearest-neighbor resize via cv2 (preserves NaN exactly because
                # nearest is index-only). Avoids the torch.from_numpy + interpolate
                # + .numpy() round-trip; same result.
                hha_hwc = np.ascontiguousarray(hha_native.transpose(1, 2, 0))
                hha_resized_hwc = cv2.resize(
                    hha_hwc, (target_size, target_size),
                    interpolation=cv2.INTER_NEAREST,
                )
                hha_resized = hha_resized_hwc.transpose(2, 0, 1)
                hha_dtype = torch.float16 if HHA_DTYPE == 'float16' else torch.float32
                hha_tensor = torch.from_numpy(hha_resized).to(hha_dtype)

            # Convert to tensors: RGB [3, H, W] uint8, depth [1, H, W] uint16
            rgb_tensor = torch.from_numpy(
                color_crop.transpose(2, 0, 1).copy()
            ).to(torch.uint8)
            depth_tensor = torch.from_numpy(
                depth_crop[np.newaxis, :, :].copy()
            ).to(torch.uint16)

            # Save all tensors together. In HHA mode, hha_tensor is guaranteed
            # not None at this point (we'd have continued/break otherwise).
            prefix = f'{scene_id}_f{idx:05d}'
            torch.save(rgb_tensor, os.path.join(output_dir, f'{prefix}_rgb.pt'))
            if HHA_MODE in ('none', 'with-hha'):
                torch.save(depth_tensor, os.path.join(output_dir, f'{prefix}_depth.pt'))
            if hha_tensor is not None:
                torch.save(hha_tensor, os.path.join(output_dir, f'{prefix}_hha.pt'))

            used_indices.add(idx)
            extracted += 1
            frame_saved = True
            if target_was_blurry:
                blur_replaced += 1
            break  # Got a sharp frame for this sample point

        if not frame_saved:
            if target_was_blurry:
                blur_dropped += 1
            elif target_pose_failed:
                pose_dropped += 1

    return extracted, blur_replaced, blur_dropped, pose_dropped, decode_failed, used_indices

## 9. Download-Extract-Delete Micro-Loop

**CRITICAL: Colab disk constraint.** Raw `.sens` files are 100-200MB each.
Each worker handles the full lifecycle for one scene:
download -> extract -> delete.

Pool capped at `MAX_WORKERS=4` to limit concurrent disk usage (~800MB peak)
and avoid server rate-limiting.

In [ ]:
def process_scene(args):
    """Full download-extract-delete micro-loop for one scene.

    args = (scene_id, split_name, txt_dir, output_base_dir,
            sens_tmp_dir, num_frames, target_size)

    Returns dict with keys: scene_id, class_name, num_frames,
    status ('ok'/'skipped'/'error'), error_msg (if error).
    """

    scene_id, split_name, txt_dir, output_base_dir, sens_tmp_dir, \
        num_frames, target_size = args

    result = {
        'scene_id': scene_id,
        'class_name': None,
        'num_frames': 0,
        'blur_replaced': 0,
        'blur_dropped': 0,
        'pose_dropped': 0,
        'decode_failed': 0,
        # Pass 2 (rare-class supplement) accounting
        'pass1_frames': 0,
        'pass2_frames': 0,
        'pass2_attempted': False,
        'pass2_error': None,
        'status': 'error',
        'error_msg': None,
    }

    done_marker = os.path.join(output_base_dir, f'{scene_id}.done')
    failed_marker = os.path.join(output_base_dir, f'{scene_id}.failed')

    # CRIT #2: .failed marker must take precedence over .done. A scene
    # that legitimately failed (e.g., HHA meta load) should NOT be re-tried
    # via the incremental-Pass-2 path just because a stale .done marker
    # exists from an earlier run.
    if os.path.exists(failed_marker):
        result['status'] = 'skipped'
        result['error_msg'] = 'permanent_failure_marker_present'
        return result

    # (1) Check .done marker — but allow Pass 2 incremental top-up for rare-class scenes
    # whose marker predates Pass 2.
    _existing_marker_lines = None
    if os.path.exists(done_marker):
        try:
            with open(done_marker) as _mf:
                _existing_marker_lines = _mf.read().splitlines()
        except OSError as _me:
            # CRIT #3: marker unreadable -> behave like no marker.
            print(f'WARNING: cannot read .done marker for {scene_id}: {_me}; treating as fresh.')
            _existing_marker_lines = None
        # Marker fields written as 'key=value' on later lines.
        # CRIT-R4 #1: guard the iteration -- _existing_marker_lines is None
        # when the read failed; skip parsing entirely (treats scene as fresh).
        _marker_kv = {}
        if _existing_marker_lines is not None:
            for _ln in _existing_marker_lines:
                if '=' in _ln:
                    _k, _v = _ln.split('=', 1)
                    _marker_kv[_k.strip()] = _v.strip()
        # If marker says Pass 2 was already attempted (succeeded or determined
        # not applicable), this scene is fully done. Skip.
        # Older markers (pre-Pass-2) lack this field -> treat as "needs Pass 2 check".
        if _marker_kv.get('pass2_attempted', '').lower() == 'true' or _marker_kv.get('pass2_done', '').lower() == 'true':
            result['status'] = 'skipped'
            return result
        # Marker exists but Pass 2 not yet attempted. We'll re-download .sens,
        # recover Pass 1 indices from existing files, and run Pass 2 only IF
        # this scene's class is rare. The default 'error' status is preserved
        # so a download failure mid-incremental properly counts as an error
        # (R5 #2: was 'incremental_pass2' which silently masked download failures
        # from the post-loop error tally).
    # Parse metadata to get scene type
    # download-scannet.py may nest files in various subdirectory structures
    # (e.g., txt_dir/scene0000_00.txt, txt_dir/scene0000_00/scene0000_00.txt,
    #  txt_dir/scans/scene0000_00/scene0000_00.txt)
    txt_candidates = [
        os.path.join(txt_dir, 'scans', scene_id, f'{scene_id}.txt'),
        os.path.join(txt_dir, f'{scene_id}.txt'),
        os.path.join(txt_dir, scene_id, f'{scene_id}.txt'),
    ]
    # Also search recursively as fallback
    txt_candidates += glob.glob(
        os.path.join(txt_dir, '**', f'{scene_id}.txt'), recursive=True
    )
    txt_path = None
    for candidate in txt_candidates:
        if os.path.exists(candidate):
            txt_path = candidate
            break
    if txt_path is None:
        result['error_msg'] = 'No .txt metadata found'
        return result

    meta = parse_scene_metadata(txt_path)
    raw_type = meta.get('sceneType', '')
    canonical = normalize_scene_type(raw_type)
    if canonical is None:
        result['status'] = 'skipped'
        result['error_msg'] = f'Unrecognized sceneType: {raw_type!r}'
        return result

    result['class_name'] = canonical
    scene_out_dir = os.path.join(output_base_dir, split_name, canonical)
    os.makedirs(scene_out_dir, exist_ok=True)

    # Incremental-mode decision: if Pass 1 was already done (existing marker)
    # but this class is NOT rare, there's nothing for Pass 2 to do. Mark the
    # marker as pass2_attempted=False (recorded as definitively-not-needed)
    # and skip cleanly without redownloading the .sens.
    # CRIT-R4 #1: only enter incremental mode if we actually parsed the marker
    # (None = unreadable; [] = empty). Either way, do NOT pretend Pass 1 succeeded.
    _is_incremental = bool(_existing_marker_lines)
    _class_is_rare = (ENABLE_PASS2 and 'PASS2_RARE_CLASSES' in globals() and canonical in PASS2_RARE_CLASSES)
    if _is_incremental and not _class_is_rare:
        # CRIT #3: only append to a non-empty marker. Empty content means
        # something went wrong reading it -- preserve whatever's on disk.
        # HIGH-R4 #8: skip the rewrite if marker already records pass2_attempted
        # (any value) -- otherwise repeated re-runs append duplicate lines forever.
        if _existing_marker_lines and 'pass2_attempted' not in _marker_kv:
            _tmp_m = done_marker + '.tmp'
            with open(_tmp_m, 'w') as _f:
                _f.write('\n'.join(_existing_marker_lines))
                _f.write('\n')  # splitlines() strips newlines, always re-add
                _f.write('pass2_attempted=False\n')
                _f.write('pass2_skipped_reason=class_not_rare\n')
                _f.flush()
                os.fsync(_f.fileno())
            os.replace(_tmp_m, done_marker)
        result['status'] = 'skipped'
        return result

    # (2) If no marker but partial files exist, wipe and re-process.
    # Preserve files when in incremental mode (Pass 1 frames are KEPT for Pass 2 to seed off).
    if not _is_incremental:
        existing_files = glob.glob(os.path.join(scene_out_dir, f'{scene_id}_f*_*.pt'))
        if existing_files:
            # R5 #12/#13: if a .done marker exists but parsed empty (corrupted/
            # truncated), and there are matching files on disk, treat them as
            # Pass 1 data and switch to incremental rather than wiping. The
            # alternative -- wiping real frames -- destroys ~85 frames + 150 MB
            # of TUM bandwidth needed to re-extract them.
            if os.path.exists(done_marker):
                print(f'WARNING: empty/corrupt .done marker for {scene_id} but '
                      f'{len(existing_files)} frame files exist; treating as incremental.')
                _is_incremental = True
                _existing_marker_lines = ['recovered_from_empty_marker']  # truthy sentinel
            else:
                for f in existing_files:
                    os.remove(f)

    # (3) Download .sens via subprocess with retry + exponential backoff
    sens_download_dir = os.path.join(sens_tmp_dir, scene_id)
    os.makedirs(sens_download_dir, exist_ok=True)
    # download-scannet.py saves to <out>/scans/<scene_id>/<scene_id>.sens
    sens_path = os.path.join(sens_download_dir, 'scans', scene_id, f'{scene_id}.sens')

    max_retries = 3
    for attempt in range(max_retries):
        cmd = [
            'python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
            '-o', sens_download_dir,
            '--id', scene_id,
            '--type', '.sens',
        ]
        try:
            # 5 min timeout is enough for a 150 MB .sens at 0.5 MB/s. If TUM is
            # slower than that, retry helps more than waiting longer.
            _r = subprocess.run(
                cmd, capture_output=True, text=True, timeout=300,
                input='y\ny\n',
            )
            # Locate the saved .sens (download script may place it in subdir).
            if not os.path.exists(sens_path):
                for alt in [
                    os.path.join(sens_download_dir, scene_id, f'{scene_id}.sens'),
                    os.path.join(sens_download_dir, f'{scene_id}.sens'),
                ]:
                    if os.path.exists(alt):
                        sens_path = alt
                        break

            # #4/#5: file existence is necessary but not sufficient. A truncated
            # or empty .sens (network drop, disk full) passes os.path.exists.
            # Validate by trying to OPEN it via SensorData; a partial file will
            # raise on header parse. Also reject 0-byte and check returncode.
            ok = False
            if os.path.exists(sens_path) and os.path.getsize(sens_path) > 0 and _r.returncode == 0:
                try:
                    _probe = SensorData.SensorData(sens_path)
                    if hasattr(_probe, 'frames') and len(_probe.frames) > 0:
                        ok = True
                    del _probe
                except Exception:
                    pass  # treat as failed download
            if ok:
                break
            else:
                # Bad file — wipe it so the retry doesn't think we already have it
                if os.path.exists(sens_path):
                    try: os.remove(sens_path)
                    except Exception: pass
        except subprocess.TimeoutExpired:
            pass
        except Exception as _se:
            # FIX #3: catch FileNotFoundError (python3 missing), OSError
            # (disk full mid-write), permission errors, and other subprocess
            # failures. Without this, the exception propagates out of the
            # worker and Pool.imap_unordered re-raises it in the main loop,
            # killing the entire extraction run mid-flight.
            if attempt == max_retries - 1:
                result['error_msg'] = f'download exception: {type(_se).__name__}: {str(_se)[:200]}'

        if attempt < max_retries - 1:
            # Backoff scaled to be MEANINGFUL relative to the timeout: 30/60/120s
            # (was 2/4/8s, which is noise for a 5-min timeout and useless against
            # TUM rate-limiting that requires backing off for tens of seconds).
            time.sleep(30 * (2 ** attempt))

    if not os.path.exists(sens_path):
        result['error_msg'] = 'Failed to download .sens after retries'
        # Cleanup
        shutil.rmtree(sens_download_dir, ignore_errors=True)
        return result

    # (4) Extract frames with 120s timeout
    try:
        # === Populate per-scene HHA cache before extraction ===
        if HHA_MODE != 'none':
            try:
                # Pull intrinsic_depth (4x4 -> 3x3) and depth_shift from .sens header.
                _sd_meta = SensorData.SensorData(sens_path)
                K_native = np.asarray(_sd_meta.intrinsic_depth, dtype=np.float64)[:3, :3]
                depth_shift = float(getattr(_sd_meta, 'depth_shift', 1000.0))
                # Read axisAlignment from <scene>.txt (the same metadata file we already parsed).
                # 'meta' is the parsed scene metadata dict from earlier; check for 'axisAlignment'.
                axis_align_4x4 = None
                if 'axisAlignment' in meta:
                    vals = np.array(meta['axisAlignment'].split(), dtype=np.float64)
                    if vals.size == 16:
                        axis_align_4x4 = vals.reshape(4, 4)
                if axis_align_4x4 is None:
                    # Missing axisAlignment -> identity fallback (drop list reason 'missing_axisAlignment')
                    axis_align_rot = np.eye(3, dtype=np.float64)
                else:
                    raw = axis_align_4x4[:3, :3]
                    if SCANNET_AXIS_INVERTED:
                        raw = raw.T  # canonicalize to 'aligned <- raw'
                    axis_align_rot = _orthogonalize(raw)
                _HHA_SCENE_META_CACHE[scene_id] = {
                    'K': K_native,
                    'axis_alignment_rot': axis_align_rot,
                    'depth_shift': depth_shift,
                }
                # Keep _sd_meta alive — passed to extract_frames_from_sens
                # below to avoid a second ~150 MB SensorData parse.
            except (ValueError, KeyError, AttributeError, OSError, IndexError, TypeError) as _e:
                # FIX #7: narrowed from `except Exception` so MemoryError,
                # KeyboardInterrupt, SystemExit propagate instead of being
                # silently converted into per-scene "permanent" failures.
                # FIX #5: meta load failed permanently. Write .failed marker so
                # subsequent runs skip the .sens redownload. Cleanup the .sens
                # we just downloaded so disk doesn't fill on bulk retries.
                _HHA_SCENE_META_CACHE.pop(scene_id, None)
                msg = f'{type(_e).__name__}: {str(_e)[:200]}'
                print(f'WARNING: HHA scene_meta load failed for {scene_id}: {msg}')
                _tmp_fail = failed_marker + '.tmp'
                with open(_tmp_fail, 'w') as _f:
                    _f.write(f'hha_meta_load_failed\n{msg}\n')
                    _f.flush()
                    os.fsync(_f.fileno())
                os.replace(_tmp_fail, failed_marker)
                shutil.rmtree(sens_download_dir, ignore_errors=True)
                result['error_msg'] = f'hha_meta_load_failed: {msg}'
                result['status'] = 'error'
                return result

        # Pass the already-parsed SensorData (if HHA mode loaded it) so
        # extract_frames_from_sens doesn't re-parse the .sens (~150 MB walk).
        _sd_for_extract = locals().get('_sd_meta') if HHA_MODE != 'none' else None
        if _is_incremental:
            # === Skip Pass 1: recover its frame indices from disk filenames. ===
            # Files are named "{scene_id}_f{idx:05d}_{rgb|hha}.pt".
            used_indices_p1 = set()
            for _f in os.listdir(scene_out_dir):
                _m = re.match(rf'{re.escape(scene_id)}_f(\d+)_rgb\.pt$', _f)
                if _m:
                    used_indices_p1.add(int(_m.group(1)))
            n_extracted = len(used_indices_p1)
            n_blur_replaced = n_blur_dropped = n_pose_dropped = n_decode_failed = 0
            result['pass1_frames']  = n_extracted
            # R5 #1: seed result['blur_replaced'] etc. from the existing marker's
            # Pass 1 counters so the rewritten marker preserves Pass 1 stats.
            # Marker lines are like "{N} blur_replaced" -- parse them.
            for _ln in (_existing_marker_lines or []):
                _ln_s = _ln.strip()
                for _key in ('blur_replaced', 'blur_dropped', 'pose_dropped', 'decode_failed'):
                    if _ln_s.endswith(' ' + _key):
                        try:
                            result[_key] = int(_ln_s.split(' ', 1)[0])
                        except ValueError:
                            pass
        else:
            # === PASS 1 (full run): evenly-spaced 85 fps ===
            n_extracted, n_blur_replaced, n_blur_dropped, n_pose_dropped, n_decode_failed, used_indices_p1 = extract_frames_from_sens(
                sens_path, scene_out_dir, scene_id,
                num_frames=num_frames, target_size=target_size,
                blur_threshold=BLUR_THRESHOLD,
                blur_search_window=BLUR_SEARCH_WINDOW,
                sd_obj=_sd_for_extract,
            )
            result['pass1_frames']  = n_extracted
            result['blur_replaced'] = n_blur_replaced
            result['blur_dropped']  = n_blur_dropped
            result['pose_dropped']  = n_pose_dropped
            result['decode_failed'] = n_decode_failed

        # === PASS 2 (only for rare-class scenes): supplement at temporal offset ===
        # Run when Pass 2 is enabled AND class is rare. Pass 1 frame count gate
        # only applies when Pass 1 just ran this call (incremental mode trusts
        # the existing marker's Pass 1 success).
        n2_extracted = 0
        _pass2_should_run = (
            ENABLE_PASS2
            and _class_is_rare
            and (_is_incremental or n_extracted >= MIN_FRAMES)
        )
        if _pass2_should_run:
            # BUG E/F FIX: set pass2_attempted=True UPFRONT so the marker
            # records "we tried" even if Pass 2 raises. Marker write at the
            # end of process_scene then prevents infinite retry. Pass 1 frames
            # are NEVER touched by this block on failure (isolated try/except).
            result['pass2_attempted'] = True
            try:
                # Build candidate frame indices that sit at >= MIN_GAP_FRAMES from
                # every Pass 1 index. Pick PASS2_TARGET evenly-spaced from those.
                _sd_for_p2 = locals().get('_sd_meta') if HHA_MODE != 'none' else None
                if _sd_for_p2 is None:
                    # Re-parse the .sens for total frame count (cheap header read).
                    _sd_for_p2 = SensorData.SensorData(sens_path)
                _total = len(_sd_for_p2.frames)
                # Build excluded set: every index within MIN_GAP_FRAMES-1 of a Pass 1 index.
                _excluded = set()
                for _p in used_indices_p1:
                    for _d in range(-MIN_GAP_FRAMES + 1, MIN_GAP_FRAMES):
                        _excluded.add(_p + _d)
                _valid = [i for i in range(_total) if i not in _excluded]
                if _valid:
                    if len(_valid) <= PASS2_TARGET:
                        _p2_targets = _valid
                    else:
                        _step = len(_valid) / PASS2_TARGET
                        _p2_targets = [_valid[int(_j * _step)] for _j in range(PASS2_TARGET)]
                    n2_extracted, n2_br, n2_bd, n2_pd, n2_df, _ = extract_frames_from_sens(
                        sens_path, scene_out_dir, scene_id,
                        num_frames=PASS2_TARGET, target_size=target_size,
                        blur_threshold=BLUR_THRESHOLD,
                        blur_search_window=BLUR_SEARCH_WINDOW,
                        sd_obj=_sd_for_p2,
                        target_indices=_p2_targets,
                        seed_used_indices=used_indices_p1,
                        min_gap=MIN_GAP_FRAMES,
                    )
                    result['pass2_frames']   = n2_extracted
                    result['blur_replaced'] += n2_br
                    result['blur_dropped']  += n2_bd
                    result['pose_dropped']  += n2_pd
                    result['decode_failed'] += n2_df
            except (OSError, ValueError, KeyError, AttributeError, IndexError, TypeError, RuntimeError) as _p2e:
                # R5 #15: narrowed from `Exception` so MemoryError /
                # KeyboardInterrupt / SystemExit propagate. Silently swallowing
                # OOM during Pass 2 lets the worker accumulate more pressure.
                # Pass 1 data is intact -- log error and continue to marker write.
                # n2_extracted stays 0; marker records pass2_attempted=True so
                # future runs skip this scene's Pass 2.
                result['pass2_error'] = f'{type(_p2e).__name__}: {str(_p2e)[:200]}'
                # Best-effort cleanup of any partial Pass 2 files: remove files
                # whose index is NOT in the original Pass 1 set.
                try:
                    for _pf in glob.glob(os.path.join(scene_out_dir, f'{scene_id}_f*_*.pt')):
                        _pm = re.match(rf'{re.escape(scene_id)}_f(\d+)_', os.path.basename(_pf))
                        if _pm and int(_pm.group(1)) not in used_indices_p1:
                            os.remove(_pf)
                except OSError:
                    pass

        # Total = Pass 1 + Pass 2
        n_extracted = n_extracted + n2_extracted
        result['num_frames'] = n_extracted
        # R5 #21: drop the ~150 MB SensorData object before the .sens file is
        # rmtree'd (line below). Avoids workers carrying a stale parse around
        # while waiting for the next task.
        try:
            del _sd_meta
        except NameError:
            pass
        try:
            del _sd_for_extract
        except NameError:
            pass
        try:
            del _sd_for_p2
        except NameError:
            pass
    except Exception as e:
        result['error_msg'] = str(e)[:200]
        # HIGH-R4 #4: ensure status is 'error' even when we entered incremental
        # mode (which had set status='incremental_pass2'). Otherwise the post-loop
        # error counter and errors[] list miss this scene entirely.
        result['status'] = 'error'
        # BUG A FIX: in incremental mode, the existing Pass 1 frames must NOT
        # be deleted -- they're valid data from a prior successful run. Only
        # delete files that aren't among the recovered Pass 1 indices.
        if _is_incremental and 'used_indices_p1' in dir():
            _keep_idx = used_indices_p1
            for _f in glob.glob(os.path.join(scene_out_dir, f'{scene_id}_f*_*.pt')):
                _m = re.match(rf'{re.escape(scene_id)}_f(\d+)_', os.path.basename(_f))
                if _m and int(_m.group(1)) not in _keep_idx:
                    os.remove(_f)
        else:
            # Fresh extraction path: wipe everything from this scene.
            partial = glob.glob(os.path.join(scene_out_dir, f'{scene_id}_f*_*.pt'))
            for f in partial:
                os.remove(f)
        shutil.rmtree(sens_download_dir, ignore_errors=True)
        return result

    # (5) Write .done marker — ONLY if extraction actually produced enough
    # frames. Without this guard, a scene where every candidate was rejected
    # (blur or invalid pose) would get marked 'done' with 0 frames and be
    # silently skipped on every future re-run.
    if n_extracted >= MIN_FRAMES:
        # Atomic write: tmp file -> fsync -> rename.
        _tmp_marker = done_marker + '.tmp'
        with open(_tmp_marker, 'w') as f:
            f.write(
                f'{canonical}\n{n_extracted}\n'
                f'{result["blur_replaced"]} blur_replaced\n'
                f'{result["blur_dropped"]} blur_dropped\n'
                f'{result["pose_dropped"]} pose_dropped\n'
                f'{result["decode_failed"]} decode_failed\n'
                f'pass1_frames={result["pass1_frames"]}\n'
                f'pass2_frames={result["pass2_frames"]}\n'
                f'pass2_attempted={result["pass2_attempted"]}\n'
                f'pass2_error={(result["pass2_error"] or "").replace(chr(10), " | ").replace(chr(13), " ")}\n'
                f'hha_mode={HHA_MODE}\n'
                f'axis_inverted={SCANNET_AXIS_INVERTED}\n'
                f'hha_dtype={HHA_DTYPE if HHA_MODE != "none" else "n/a"}\n'
            )
            f.flush()
            os.fsync(f.fileno())
        os.replace(_tmp_marker, done_marker)
        # R5 #3: surface Pass 2 failure in status so the post-loop error scan
        # catches it. Pass 1 data is intact; status reflects partial success.
        if result.get('pass2_error'):
            result['status'] = 'ok_pass2_failed'
        else:
            result['status'] = 'ok'
    else:
        # MED-R4 #30: write a .failed marker so subsequent re-runs skip the
        # 150 MB .sens redownload for permanently-bad scenes (e.g., camera
        # tracking lost, every candidate blurry). Delete .failed manually to
        # retry. Without this, every re-run wastes bandwidth on dead scenes.
        for f in glob.glob(os.path.join(scene_out_dir, f'{scene_id}_f*_*.pt')):
            os.remove(f)
        try:
            _tmp_fail = failed_marker + '.tmp'
            with open(_tmp_fail, 'w') as _ff:
                _ff.write(
                    f'extract_below_min_frames\n'
                    f'extracted={n_extracted} min={MIN_FRAMES}\n'
                    f'pass1_frames={result["pass1_frames"]}\n'
                    f'pass2_frames={result["pass2_frames"]}\n'
                )
                _ff.flush()
                os.fsync(_ff.fileno())
            os.replace(_tmp_fail, failed_marker)
        except OSError:
            pass
        result['status'] = 'error'
        result['error_msg'] = f'only {n_extracted} frames extracted (min={MIN_FRAMES}); .failed marker written'

    # (6) Delete .sens file immediately
    shutil.rmtree(sens_download_dir, ignore_errors=True)

    return result

if 'SKIP_EXTRACT' in dir() and SKIP_EXTRACT:
    print('SKIPPED: extraction loop (data restored from Drive in cell 31)')
else:
    # === Build PASS2_RARE_CLASSES set from cell 22's mapped_counts ===
    # A class is "rare" if it has < PASS2_RARE_THRESHOLD scenes total in the
    # train+val splits. Those classes get the Pass 2 supplement.
    if ENABLE_PASS2 and 'mapped_counts' in dir():
        PASS2_RARE_CLASSES = {
            cls for cls, n in mapped_counts.items()
            if n < PASS2_RARE_THRESHOLD
        }
        print(f'Pass 2 enabled. Rare classes (<{PASS2_RARE_THRESHOLD} scenes) '
              f'getting supplement: {len(PASS2_RARE_CLASSES)} classes')
        for _c in sorted(PASS2_RARE_CLASSES):
            print(f'  - {_c}: {mapped_counts[_c]} scenes')
    elif ENABLE_PASS2:
        # MED-R4 #9: was a silent fallback to set() which made every scene
        # look "non-rare" -- silently disabling Pass 2 entirely. Raise loudly.
        raise RuntimeError(
            'ENABLE_PASS2=True but mapped_counts not in scope. '
            'Re-run cell 22 (metadata pre-scan) first to populate it.'
        )
    else:
        PASS2_RARE_CLASSES = set()

    # Build task list: ONLY scene IDs from train + val splits
    train_set = set(train_scene_ids)
    val_set = set(val_scene_ids)

    # Apply Phase 0 drop list (HHA mode only). NOTE: cell 14 hard-drops
    # missing_axisAlignment scenes by adding them to SCANNET_DROP_SCENES,
    # so this filter removes them from the task list -- they are NOT
    # processed with an identity-rotation fallback.
    if HHA_MODE != 'none' and SCANNET_DROP_SCENES:
        train_scene_ids = [s for s in train_scene_ids if s not in SCANNET_DROP_SCENES]
        val_scene_ids = [s for s in val_scene_ids if s not in SCANNET_DROP_SCENES]
        print(f'After drop-list filter: train={len(train_scene_ids)}, val={len(val_scene_ids)}')

    tasks = []
    for scene_id in train_scene_ids:
        tasks.append((
            scene_id, 'train', TXT_DIR, BASE_OUT_DIR, TMP_DIR,
            FRAMES_PER_SCENE, TARGET_SIZE
        ))
    for scene_id in val_scene_ids:
        tasks.append((
            scene_id, 'val', TXT_DIR, BASE_OUT_DIR, TMP_DIR,
            FRAMES_PER_SCENE, TARGET_SIZE
        ))

    print(f'Total tasks: {len(tasks)}')

    # Process with multiprocessing pool
    progress_path = os.path.join(BASE_OUT_DIR, 'progress.json')
    results_log = []
    counts = Counter()  # ok, skipped, error
    running_frames = 0
    running_blur_replaced = 0
    running_blur_dropped = 0
    # HIGH #8: maintain pose_dropped as an incremental counter instead of
    # summing the entire results_log on every loop iteration (was O(N^2)).
    running_pose_dropped = 0

    # Pin fork start method (Linux/Colab default but can flip elsewhere).
    # Workers inherit notebook globals (process_scene, _HHA_SCENE_META_CACHE).
    with multiprocessing.get_context('fork').Pool(MAX_WORKERS) as pool:
        pbar = tqdm(total=len(tasks), desc='Processing scenes')
        # R6 #2: wrap the iteration -- pool.imap_unordered re-raises any
        # uncaught worker exception in the main loop, which would discard
        # ALL completed-but-unsaved results. Catch and log; let the user
        # diagnose from processing_log.jsonl.
        _pool_iter = pool.imap_unordered(process_scene, tasks)
        while True:
            try:
                result = next(_pool_iter)
            except StopIteration:
                break
            except Exception as _pe:
                # A worker died; record sentinel result and continue.
                result = {
                    'scene_id': '<worker_crash>',
                    'class_name': None,
                    'num_frames': 0,
                    'pass1_frames': 0,
                    'pass2_frames': 0,
                    'pass2_attempted': False,
                    'pass2_error': None,
                    'status': 'error',
                    'error_msg': f'worker exception: {type(_pe).__name__}: {str(_pe)[:200]}',
                }
            results_log.append(result)
            counts[result['status']] += 1
            running_frames += result.get('num_frames', 0)
            running_blur_replaced += result.get('blur_replaced', 0)
            running_blur_dropped += result.get('blur_dropped', 0)
            running_pose_dropped += result.get('pose_dropped', 0)

            # Update progress bar
            pbar.set_postfix(
                ok=counts['ok'], skip=counts['skipped'], err=counts['error'],
                frames=running_frames,
                blur_fix=running_blur_replaced, blur_lost=running_blur_dropped,
                pose_lost=running_pose_dropped,
            )
            pbar.update(1)

            # Write progress.json from main process only
            if len(results_log) % 10 == 0:
                with open(progress_path, 'w') as f:
                    json.dump({
                        'completed': len(results_log),
                        'total': len(tasks),
                        'ok': counts['ok'],
                        'skipped': counts['skipped'],
                        'error': counts['error'],
                    }, f)
        pbar.close()

    # Final progress write
    with open(progress_path, 'w') as f:
        json.dump({
            'completed': len(results_log),
            'total': len(tasks),
            'ok': counts['ok'],
            'skipped': counts['skipped'],
            'error': counts['error'],
        }, f)

    # #37: persist per-scene results so failures can be diagnosed post-hoc.
    log_path = os.path.join(BASE_OUT_DIR, 'processing_log.jsonl')
    with open(log_path, 'w') as f:
        for r in results_log:
            f.write(json.dumps(r) + '\n')
    print(f'  Per-scene log: {log_path}')

    total_blur_replaced = sum(r.get('blur_replaced', 0) for r in results_log)
    total_blur_dropped = sum(r.get('blur_dropped', 0) for r in results_log)
    total_pose_dropped = sum(r.get('pose_dropped', 0) for r in results_log)
    total_decode_failed = sum(r.get('decode_failed', 0) for r in results_log)
    total_frames = sum(r.get('num_frames', 0) for r in results_log)
    total_pass1 = sum(r.get('pass1_frames', 0) for r in results_log)
    total_pass2 = sum(r.get('pass2_frames', 0) for r in results_log)
    total_p2_attempted = sum(1 for r in results_log if r.get('pass2_attempted'))

    # BUG B FIX: mark dataset changed when Pass 2 added frames OR when any
    # newly-extracted scenes wrote frames in this run. Without this, cell 49
    # (norm_stats) sees CHANGES_MADE=False on a restored-then-Pass-2'd dataset
    # and skips the recompute, leaving stats blind to the new HHA frames.
    # R7: include 'ok_pass2_failed' so a run where every rare-class scene's
    # Pass 2 OOMs still flips CHANGES_MADE -> norm-stats recompute triggers.
    _new_frames_this_run = total_pass2 + sum(
        r.get('pass1_frames', 0) for r in results_log
        if r.get('status') in ('ok', 'ok_pass2_failed')
    )
    if _new_frames_this_run > 0:
        CHANGES_MADE = True

    print(f'\n=== Processing Complete ===')
    print(f'OK:      {counts["ok"]}  (newly extracted this run)')
    print(f'Skipped: {counts["skipped"]}  (already had .done marker)')
    print(f'Errors:  {counts["error"]}')
    if counts['ok'] > 0:
        print(f'Frames newly extracted: {total_frames:,}')
    else:
        # All scenes were already .done — count files on disk so the summary
        # doesn't read "0 frames extracted" when the dataset is actually full.
        _on_disk = sum(1 for _ in glob.glob(
            os.path.join(BASE_OUT_DIR, '*', '*', '*_rgb.pt')))
        print(f'Frames newly extracted: 0  '
              f'(dataset already complete: {_on_disk:,} _rgb.pt on disk)')
    print(f'Blur replaced:    {total_blur_replaced:,} (found sharp neighbor)')
    print(f'Blur dropped:     {total_blur_dropped:,} (no sharp frame in window, sample lost)')
    print(f'Pose dropped:     {total_pose_dropped:,} (HHA mode: all candidates had invalid pose)')
    print(f'Decode failed:    {total_decode_failed:,} (color/depth JPEG/zlib decompress raised)')
    print(f'Pass 1 frames:    {total_pass1:,}  (baseline 85 fps for all scenes)')
    print(f'Pass 2 frames:    {total_pass2:,}  '
          f'(supplement on {total_p2_attempted} rare-class scenes; min_gap={MIN_GAP_FRAMES})')

    # Show errors
    errors = [r for r in results_log if r['status'] == 'error']
    if errors:
        print(f'\n=== Errors ({len(errors)}) ===')
        for r in errors[:20]:
            print(f'  {r["scene_id"]}: {r["error_msg"]}')
        if len(errors) > 20:
            print(f'  ... and {len(errors) - 20} more')

    # R5 #3: separately surface Pass 2 failures (Pass 1 data preserved).
    p2_failed = [r for r in results_log if r.get('status') == 'ok_pass2_failed']
    if p2_failed:
        print(f'\n=== Pass 2 failures ({len(p2_failed)}, Pass 1 data intact) ===')
        for r in p2_failed[:20]:
            print(f'  {r["scene_id"]}: {r.get("pass2_error", "?")}')
        if len(p2_failed) > 20:
            print(f'  ... and {len(p2_failed) - 20} more')

## 9c. Silent-Drop Diagnostic

Reports scenes that have `.done` markers but no extracted files. After the cell 24 fix this should always be 0; this cell is a hard sanity check on the existing dataset state.

In [ ]:
if 'SKIP_EXTRACT' in dir() and SKIP_EXTRACT:
    print('SKIPPED: silent-drop diagnostic (data restored from Drive in cell 31)')
else:
    # Sanity: every .done marker must have at least one paired sample on disk.
    import re as _re

    _done_scenes = {os.path.basename(p).replace('.done', '')
                    for p in glob.glob(os.path.join(BASE_OUT_DIR, '*.done'))}
    _have_files = set()
    for _sp in ('train', 'val'):
        _split_dir = os.path.join(BASE_OUT_DIR, _sp)
        if not os.path.isdir(_split_dir):
            continue
        for _cls in os.listdir(_split_dir):
            _cls_dir = os.path.join(_split_dir, _cls)
            if not os.path.isdir(_cls_dir):
                continue
            for _f in os.listdir(_cls_dir):
                _m = _re.match(r'(scene\d+_\d+)_f', _f)
                if _m:
                    _have_files.add(_m.group(1))

    _silent_drops = sorted(_done_scenes - _have_files)
    print(f'Scenes with .done markers but no saved files: {len(_silent_drops)}')
    if _silent_drops:
        # #38: persist the FULL list (was: print first 20 then '...and N more').
        _drops_path = os.path.join(BASE_OUT_DIR, 'silent_drops.txt')
        with open(_drops_path, 'w') as _f:
            for _s in _silent_drops:
                _f.write(_s + '\n')
        print(f'  Full list ({len(_silent_drops)} scenes) saved to {_drops_path}')
        print('  Sample (first 20):')
        for _s in _silent_drops[:20]:
            print(f'    {_s}')
        print(f'\n  To retry: delete the .done markers listed in {_drops_path} '
              f'and re-run cell 24.')
    else:
        print('  OK: every .done marker has saved files.')

## 9b. CHECKPOINT — Snapshot Extracted Data to Drive

**Run this immediately after cell 22 completes.** It tarballs `BASE_OUT_DIR` to
Drive so a runtime crash later doesn't force you to re-download/re-extract from
`.sens` (the 3-8 hour part). The snapshot is named distinctly from the final
tarball so it doesn't collide with cell 51's upload.

Default ON (`SNAPSHOT_AFTER_EXTRACT = True`). Set False to skip only if you
genuinely don't care about losing the extraction work.

In [ ]:
if 'SKIP_EXTRACT' in dir() and SKIP_EXTRACT:
    print('SKIPPED: snapshot after extract (data restored from Drive in cell 31)')
else:
    SNAPSHOT_AFTER_EXTRACT = True

    if SNAPSHOT_AFTER_EXTRACT:
        train_dir = os.path.join(BASE_OUT_DIR, 'train')
        if not os.path.isdir(train_dir) or not any(os.scandir(train_dir)):
            print(f'No data at {train_dir} -> nothing to snapshot. Skip.')
        else:
            ensure_pigz()
            snapshot_name = f'{HHA_TARBALL_NAME.replace(".tar.gz", "")}.snapshot.tar.gz'
            snapshot_local = f'/content/{snapshot_name}'

            # Compress.
            print(f'Compressing {BASE_OUT_DIR} -> {snapshot_local} (pigz, {MAX_WORKERS} threads)...')
            subprocess.run(
                ['tar', '-I', f'pigz -p {CPU_WORKERS}', '-cf', snapshot_local,
                 '-C', os.path.dirname(BASE_OUT_DIR), os.path.basename(BASE_OUT_DIR)],
                check=True,
            )
            local_size = os.path.getsize(snapshot_local)
            print(f'Local snapshot: {local_size / 1024**3:.2f} GB')

            # Upload via Drive REST API (synchronous, verified on Google's servers).
            upload_to_drive_sync(snapshot_local, 'datasets', snapshot_name)

            os.remove(snapshot_local)
            print('CHECKPOINT SAVED. Crash recovery available via cell 27.')
    else:
        print('SNAPSHOT_AFTER_EXTRACT=False -> skipping. Runtime crash will lose all extraction work.')

In [10]:
!df -h /content
!du -sh {BASE_OUT_DIR}

Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   56G  171G  25% /
34G	/content/scannet_pretrain_256


## 10a. Verify Split Directories

The micro-loop already places tensors into `train/<class>/` and
`val/<class>/` subdirectories. This cell verifies the structure.

In [11]:
# Verify directory structure and split summary
split_summary = {}
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        print(f'WARNING: {split_dir} does not exist!')
        continue
    classes = sorted([d for d in os.listdir(split_dir)
                      if os.path.isdir(os.path.join(split_dir, d))])
    total_frames = 0
    class_counts = {}
    scene_ids = set()
    print(f'\n{split}/ ({len(classes)} classes):')
    for cls in classes:
        cls_dir = os.path.join(split_dir, cls)
        rgb_files = glob.glob(os.path.join(cls_dir, '*_rgb.pt'))
        class_counts[cls] = len(rgb_files)
        total_frames += len(rgb_files)
        for f in rgb_files:
            match = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt', os.path.basename(f))
            if match:
                scene_ids.add(match.group(1))
        print(f'  {cls:30s} {len(rgb_files):6d} frames')
    print(f'  {"TOTAL":30s} {total_frames:6d} frames  ({len(scene_ids)} scenes)')
    split_summary[split] = {
        'frames': total_frames,
        'scenes': len(scene_ids),
        'classes': len(classes),
    }

# Overall summary
print('\n' + '=' * 60)
print('DATASET SPLIT SUMMARY')
print('=' * 60)
grand_total_frames = sum(s['frames'] for s in split_summary.values())
grand_total_scenes = sum(s['scenes'] for s in split_summary.values())
for split, info in split_summary.items():
    frame_pct = 100.0 * info['frames'] / grand_total_frames if grand_total_frames > 0 else 0
    scene_pct = 100.0 * info['scenes'] / grand_total_scenes if grand_total_scenes > 0 else 0
    print(f'  {split:5s}: {info["frames"]:7,} frames ({frame_pct:5.1f}%)  |  '
          f'{info["scenes"]:5,} scenes ({scene_pct:5.1f}%)  |  '
          f'{info["classes"]} classes')
print(f'  {"total":5s}: {grand_total_frames:7,} frames            |  '
      f'{grand_total_scenes:5,} scenes            |')
print()
if grand_total_frames < 100000:
    print(f'  WARNING: {grand_total_frames:,} frames is below 100K target!')
    needed = 100000 - grand_total_frames
    print(f'  Need ~{needed:,} more frames. Consider increasing FRAMES_PER_SCENE.')
else:
    print(f'  Dataset size: {grand_total_frames:,} frames (above 100K target)')


train/ (20 classes):
  apartment                        2403 frames
  bathroom                        10290 frames
  bedroom                         16057 frames
  bookstore_library                3721 frames
  classroom                        2191 frames
  closet                            633 frames
  computer_cluster                  510 frames
  conference_room                  5893 frames
  copy_mail_room                   2642 frames
  dining_room                       752 frames
  game_room                         796 frames
  gym                               351 frames
  hallway                          2173 frames
  kitchen                          6294 frames
  laundry_room                     1106 frames
  living_room                     14661 frames
  lobby                            3207 frames
  office                          10908 frames
  stairs                            677 frames
  storage_basement_garage           815 frames
  TOTAL                           8608

## 10b. Rebalance Split (79/21 -> 90/10)

The official ScanNet split is 79/21 (train/val). For pretraining we want
90/10 — more training data, minimal val for monitoring. This cell moves
scenes from val to train, stratified by class, keeping at least 1 scene
per class in val.

In [ ]:
# Target: 90% train / 10% val by frame count
TARGET_VAL_FRACTION = 0.10

# Count current frames per scene per split
scene_info = {}  # scene_id -> {'split': str, 'class': str, 'frames': int}
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        # Group files by scene_id
        scene_files = defaultdict(list)
        for f in os.listdir(cls_dir):
            if f.endswith('_rgb.pt'):
                match = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt', f)
                if match:
                    scene_files[match.group(1)].append(f)
        for sid, files in scene_files.items():
            scene_info[sid] = {'split': split, 'class': cls, 'frames': len(files)}

total_frames = sum(s['frames'] for s in scene_info.values())
current_val_frames = sum(s['frames'] for s in scene_info.values() if s['split'] == 'val')
target_val_frames = int(total_frames * TARGET_VAL_FRACTION)

print(f'Current split: train={total_frames - current_val_frames:,} / val={current_val_frames:,} '
      f'({100 * current_val_frames / total_frames:.1f}% val)')
print(f'Target:  ~{total_frames - target_val_frames:,} / ~{target_val_frames:,} '
      f'({TARGET_VAL_FRACTION * 100:.0f}% val)')
print(f'Need to move ~{current_val_frames - target_val_frames:,} frames from val to train')

# Group val scenes by class
val_scenes_by_class = defaultdict(list)
for sid, info in scene_info.items():
    if info['split'] == 'val':
        val_scenes_by_class[info['class']].append((sid, info['frames']))

# Sort each class's val scenes by frame count (move largest first to reach target faster)
for cls in val_scenes_by_class:
    val_scenes_by_class[cls].sort(key=lambda x: -x[1])

# Greedily move val scenes to train until we hit the target,
# but always keep at least 1 scene per class in val
scenes_to_move = []
frames_to_move = 0
frames_needed = current_val_frames - target_val_frames

# Round-robin across classes to keep the move balanced
moved_per_class = Counter()
keep_going = True
while keep_going and frames_to_move < frames_needed:
    keep_going = False
    for cls in sorted(val_scenes_by_class.keys()):
        remaining = len(val_scenes_by_class[cls]) - moved_per_class[cls]
        if remaining <= 2:  # Keep at least 2 scenes per class in val (was 1)
            continue
        idx = moved_per_class[cls]
        sid, n_frames = val_scenes_by_class[cls][idx]
        scenes_to_move.append((sid, cls, n_frames))
        frames_to_move += n_frames
        moved_per_class[cls] += 1
        keep_going = True
        if frames_to_move >= frames_needed:
            break

print(f'\nMoving {len(scenes_to_move)} scenes ({frames_to_move:,} frames) from val to train')

# Execute the moves (tqdm so the user sees progress on slow disks).
moved = 0
for sid, cls, n_frames in tqdm(scenes_to_move, desc='Rebalancing val->train'):
    val_cls_dir = os.path.join(BASE_OUT_DIR, 'val', cls)
    train_cls_dir = os.path.join(BASE_OUT_DIR, 'train', cls)
    os.makedirs(train_cls_dir, exist_ok=True)
    for f in os.listdir(val_cls_dir):
        if f.startswith(sid + '_f'):
            shutil.move(os.path.join(val_cls_dir, f), os.path.join(train_cls_dir, f))
    moved += 1

print(f'Moved {moved} scenes')
REBALANCE_MOVED = moved  # exposed to cell 51 metadata writer
if moved > 0:
    CHANGES_MADE = True
    print(f'  WARNING: official ScanNet train/val split has been modified.')
    print(f'  Benchmark numbers from papers using the canonical split are NOT '
          f'directly comparable. {moved} scenes moved from val -> train.')

# Show new split summary
print(f'\n{"=" * 60}')
print('NEW SPLIT SUMMARY')
print(f'{"=" * 60}')
new_split = {}
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    total = 0
    scenes = set()
    classes = set()
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        rgb_files = glob.glob(os.path.join(cls_dir, '*_rgb.pt'))
        if rgb_files:
            classes.add(cls)
        total += len(rgb_files)
        for f in rgb_files:
            match = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt', os.path.basename(f))
            if match:
                scenes.add(match.group(1))
    new_split[split] = {'frames': total, 'scenes': len(scenes), 'classes': len(classes)}

grand_total = sum(s['frames'] for s in new_split.values())
for split, info in new_split.items():
    pct = 100.0 * info['frames'] / grand_total if grand_total > 0 else 0
    print(f'  {split:5s}: {info["frames"]:7,} frames ({pct:5.1f}%)  |  '
          f'{info["scenes"]:5,} scenes  |  {info["classes"]} classes')
print(f'  {"total":5s}: {grand_total:7,} frames')

# Verify all classes represented in val
val_dir = os.path.join(BASE_OUT_DIR, 'val')
val_classes = [d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))
               and glob.glob(os.path.join(val_dir, d, '*_rgb.pt'))]
missing = set(SCANNET_SCENE_TYPES) - set(val_classes)
if missing:
    print(f'\n  WARNING: Val missing classes: {sorted(missing)}')
else:
    print(f'\n  All {len(val_classes)} classes represented in val')

# HIGH #6: persist rebalance provenance so cell 51 reports correctly even
# after a fresh restore (where cell 35 didn't run this session).
try:
    _rb_path = os.path.join(BASE_OUT_DIR, 'rebalance.json')
    with open(_rb_path, 'w') as _rbf:
        json.dump({'rebalance_moved': int(REBALANCE_MOVED)}, _rbf)
except (OSError, NameError):
    pass


## 11. Validate Tensor Files

Scan all `.pt` files, verify shapes (RGB: `[3,256,256]` uint8,
Depth: `[1,256,256]` uint16), pairing, and delete corrupt files.

In [ ]:
# === Auto-cleanup of orphan / corrupt files. ===
# DELETE_UNPAIRED=True is the run-all default: orphans (RGB without HHA
# companion or vice-versa) and corrupt files (torch.load fails) are deleted
# automatically. Cell 24 (the new fix) prevents most orphans at extraction
# time, so the deletions here should be small if any.
#
# Set DELETE_UNPAIRED=False if you want to inspect first (manual mode).
DELETE_UNPAIRED = True   # auto-clean any extraction-time defects (orphans, corrupt). Set False if you want to inspect first.

# Companion file extension depends on HHA mode.
COMPANION_EXT = '_hha.pt' if HHA_MODE != 'none' else '_depth.pt'
print(f'Pairing _rgb.pt with {COMPANION_EXT} (HHA_MODE={HHA_MODE})')


def _validate_one(args):
    """Validate one (rgb, companion) pair. Returns ('ok', None) | ('unpaired', rgb_path) | ('corrupt', (path, err))."""
    rgb_path, comp_ext, target_size = args
    comp_path = rgb_path.replace('_rgb.pt', comp_ext)
    if not os.path.exists(comp_path):
        return ('unpaired', rgb_path)
    try:
        rgb = torch.load(rgb_path, weights_only=True)
        comp = torch.load(comp_path, weights_only=True)
        assert rgb.shape == (3, target_size, target_size), f'RGB shape {rgb.shape}'
        assert rgb.dtype == torch.uint8, f'RGB dtype {rgb.dtype}'
        if comp_ext == '_depth.pt':
            assert comp.shape == (1, target_size, target_size), f'Depth shape {comp.shape}'
            assert comp.dtype in (torch.int16, torch.uint16), f'Depth dtype {comp.dtype}'
        else:
            assert comp.shape == (3, target_size, target_size), f'HHA shape {comp.shape}'
            assert comp.dtype in (torch.float16, torch.float32), f'HHA dtype {comp.dtype}'
        return ('ok', None)
    except Exception as e:
        return ('corrupt', (rgb_path, str(e)))


# Collect all rgb paths + orphan companion paths in a single sweep.
print('Scanning files...')
all_rgb_paths = []
orphan_comps = []  # _hha.pt or _depth.pt without paired _rgb.pt
_tmp_leaks = []     # #15: .tmp files left by interrupted atomic writes
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.isdir(split_dir):
        continue
    for cls_entry in os.scandir(split_dir):
        if not cls_entry.is_dir():
            continue
        names = set(os.listdir(cls_entry.path))
        for name in names:
            if name.endswith('_rgb.pt'):
                all_rgb_paths.append(os.path.join(cls_entry.path, name))
            elif name.endswith(COMPANION_EXT):
                rgb_name = name.replace(COMPANION_EXT, '_rgb.pt')
                if rgb_name not in names:
                    orphan_comps.append(os.path.join(cls_entry.path, name))
            elif name.endswith('.tmp'):
                # #15: leftover from interrupted atomic save (cell 39 or
                # cell 26). Always safe to delete.
                _tmp_leaks.append(os.path.join(cls_entry.path, name))

if _tmp_leaks:
    print(f'  Found {len(_tmp_leaks)} .tmp leak(s) — deleting.')
    for _tp in _tmp_leaks:
        try: os.remove(_tp)
        except OSError: pass

print(f'  RGB files: {len(all_rgb_paths)}')
print(f'  Orphan {COMPANION_EXT}: {len(orphan_comps)}')

# FIX #2: in HHA mode, orphan _hha.pt files are recoverable -- cell 39
# re-downloads the .sens to extract the missing _rgb.pt. Deleting them here
# would make cell 39 a permanent no-op. Keep them; cell 39 handles them.
# In non-HHA mode, orphan _depth.pt is unrecoverable (we don't recompute
# depth from raw frames), so it's still safe to drop.
HHA_RECOVERABLE = (HHA_MODE != 'none' and COMPANION_EXT == '_hha.pt')
if HHA_RECOVERABLE and orphan_comps:
    print(f'  -> {len(orphan_comps)} orphan _hha.pt files DEFERRED to cell 39 '
          f'(RGB recovery). Not deleting them in this cell.')
    deferred_orphans = list(orphan_comps)
    orphan_comps = []  # don't queue for deletion
else:
    deferred_orphans = []

# Parallel validate.
tasks = [(p, COMPANION_EXT, TARGET_SIZE) for p in all_rgb_paths]
checked = 0
unpaired = list(orphan_comps)
corrupt_files = []

print(f'Validating with {MAX_WORKERS} processes...')
with multiprocessing.Pool(MAX_WORKERS) as pool:
    for status, payload in tqdm(
        pool.imap_unordered(_validate_one, tasks, chunksize=64),
        total=len(tasks), desc='Validating',
    ):
        if status == 'ok':
            checked += 1
        elif status == 'unpaired':
            unpaired.append(payload)
        else:  # corrupt
            corrupt_files.append(payload)

print(f'Validated: {checked} frame pairs')

if unpaired:
    print(f'\nUnpaired files ({len(unpaired)}):')
    for p in unpaired[:10]:
        print(f'  {p}')
    if len(unpaired) > 10:
        print(f'  ...and {len(unpaired) - 10} more')
    if DELETE_UNPAIRED:
        print(f'DELETE_UNPAIRED=True -> deleting {len(unpaired)} files...')
        for p in unpaired:
            os.remove(p)
    else:
        print('DELETE_UNPAIRED=False -> NOT deleting. Review above and re-run with DELETE_UNPAIRED=True if these are real orphans.')

if corrupt_files:
    print(f'\nCorrupt files ({len(corrupt_files)}):')
    for path, err in corrupt_files[:10]:
        print(f'  {path}: {err}')
    if DELETE_UNPAIRED:
        print(f'DELETE_UNPAIRED=True -> deleting {len(corrupt_files)} corrupt files + companions...')
        for path, _ in corrupt_files:
            if os.path.exists(path):
                os.remove(path)
            comp_path = path.replace('_rgb.pt', COMPANION_EXT)
            if os.path.exists(comp_path):
                os.remove(comp_path)
    else:
        print('DELETE_UNPAIRED=False -> NOT deleting corrupt files.')

if not unpaired and not corrupt_files:
    print('All files valid.')
if DELETE_UNPAIRED and (unpaired or corrupt_files):
    CHANGES_MADE = True

## 11b. Recover Missing _rgb.pt (One-Shot Repair)

If a previous run deleted `_rgb.pt` files but `_hha.pt` files survived (e.g.,
because the validation cell paired against the wrong companion), this cell
re-extracts ONLY the missing RGB tensors at the exact frame indices recorded
in the existing `_hha.pt` filenames. It does NOT touch existing HHA tensors
and does NOT re-run blur rejection.

**Set `RUN_RGB_RECOVERY = True` to enable. Leave False during normal runs.**

In [ ]:
RUN_RGB_RECOVERY = True   # no-op if nothing to recover; auto-fixes any RGB lost to bad poses or interrupted writes


# Top-level worker for multiprocessing.Pool. Globals (subprocess, os,
# SensorData, cv2, torch, shutil) are fork-inherited from the notebook.
def _recover_rgb_for_scene(args):
    sid, scene_out_dir, indices, recovery_tmp, tools_dir, target_size = args
    sens_dl = os.path.join(recovery_tmp, sid)
    os.makedirs(sens_dl, exist_ok=True)
    sens_path = os.path.join(sens_dl, 'scans', sid, f'{sid}.sens')
    written = 0
    decode_failed = 0
    try:
        if not os.path.exists(sens_path):
            cmd = ['python3', os.path.join(tools_dir, 'download-scannet.py'),
                   '-o', sens_dl, '--id', sid, '--type', '.sens']
            try:
                subprocess.run(cmd, capture_output=True, text=True,
                               timeout=600, input='y\ny\n')
            except subprocess.TimeoutExpired:
                # FIX #3: per-scene timeout. Returning here lets the pool
                # continue with other scenes instead of crashing the run.
                return (sid, 'download_timeout', 0, len(indices))
            except Exception as _se:
                # FIX #3: any other subprocess error (FileNotFoundError on
                # python3, OSError on disk full, etc.) -- worker would die
                # otherwise and Pool.imap_unordered would re-raise in main.
                return (sid, f'download_error:{type(_se).__name__}', 0, len(indices))
        if not os.path.exists(sens_path):
            return (sid, 'download_failed', 0, len(indices))

        try:
            sd = SensorData.SensorData(sens_path)
        except Exception as _se:
            # FIX #3: SensorData parse can raise on truncated/corrupted .sens.
            return (sid, f'parse_error:{type(_se).__name__}', 0, len(indices))
        depth_h, depth_w = sd.depth_height, sd.depth_width
        for idx in indices:
            if idx >= len(sd.frames):
                continue
            out_path = os.path.join(scene_out_dir, f'{sid}_f{idx:05d}_rgb.pt')
            if os.path.exists(out_path):
                continue
            try:
                color_img = sd.frames[idx].decompress_color(sd.color_compression_type)
                if color_img is None:
                    continue
                if color_img.shape[:2] != (depth_h, depth_w):
                    color_img = cv2.resize(color_img, (depth_w, depth_h),
                                           interpolation=cv2.INTER_AREA)
                color_crop = cv2.resize(color_img, (target_size, target_size),
                                        interpolation=cv2.INTER_AREA)
                rgb_tensor = torch.from_numpy(
                    color_crop.transpose(2, 0, 1).copy()
                ).to(torch.uint8)
                # Atomic save: .tmp -> fsync -> rename. Prevents zero-byte
                # files if the worker is killed mid-write.
                tmp_out = out_path + '.tmp'
                torch.save(rgb_tensor, tmp_out)
                _fd = os.open(tmp_out, os.O_RDONLY)
                os.fsync(_fd)
                os.close(_fd)
                os.rename(tmp_out, out_path)
                if os.path.getsize(out_path) > 0:
                    written += 1
            except Exception:
                decode_failed += 1
                continue
        # Surface decode failures so they can be tracked.
        if decode_failed > 0:
            return (sid, f'partial_decode_failed_{decode_failed}', written, len(indices))
        return (sid, 'ok', written, len(indices))
    finally:
        shutil.rmtree(sens_dl, ignore_errors=True)


if RUN_RGB_RECOVERY:
    if HHA_MODE == 'none':
        raise RuntimeError('Recovery only applies to HHA mode (no _hha.pt files in HHA_MODE=none).')

    # 1. Build (scene_id -> [missing frame indices]) from existing _hha.pt files.
    # Sanity: BASE_OUT_DIR must be on local disk (NOT /content/drive — Drive's FUSE
    # is 100-1000x slower for many-file scans). Fail fast otherwise.
    assert not BASE_OUT_DIR.startswith('/content/drive'), (
        f'BASE_OUT_DIR points to Drive ({BASE_OUT_DIR}). Per-file scans on Drive are extremely slow. '
        f'Fix BASE_OUT_DIR to /content/... or /dev/shm/... in cell 4 first.'
    )
    print(f'Scanning {BASE_OUT_DIR} for _hha.pt files...')

    import time as _time
    _t0 = _time.time()

    scene_to_dir = {}
    scene_to_idx = {}
    total_hha = 0
    total_missing_rgb = 0
    for split in ['train', 'val']:
        split_dir = os.path.join(BASE_OUT_DIR, split)
        if not os.path.isdir(split_dir):
            print(f'  {split}/: missing, skipping')
            continue
        # os.scandir is faster than listdir + per-entry stat.
        cls_entries = [e for e in os.scandir(split_dir) if e.is_dir()]
        print(f'  {split}/: {len(cls_entries)} classes')
        for cls_entry in cls_entries:
            cls_dir = cls_entry.path
            _t1 = _time.time()
            # Single listdir, then set-membership check (O(1) instead of stat per file).
            names = set(os.listdir(cls_dir))
            n_hha = 0
            n_missing = 0
            for fname in names:
                m = re.match(r'(scene\d+_\d+)_f(\d+)_hha\.pt$', fname)
                if not m:
                    continue
                n_hha += 1
                sid, idx = m.group(1), int(m.group(2))
                rgb_name = f'{sid}_f{idx:05d}_rgb.pt'
                if rgb_name in names:
                    continue
                n_missing += 1
                scene_to_dir[sid] = cls_dir
                scene_to_idx.setdefault(sid, []).append(idx)
            total_hha += n_hha
            total_missing_rgb += n_missing
            print(f'    {cls_entry.name:30s} {n_hha:6d} hha  {n_missing:6d} missing-rgb  ({_time.time()-_t1:.2f}s)')

    print(f'Scan took {_time.time()-_t0:.1f}s')
    print(f'Total _hha.pt files:     {total_hha}')
    print(f'Total missing _rgb.pt:   {total_missing_rgb}')

    n_scenes = len(scene_to_idx)
    n_missing = sum(len(v) for v in scene_to_idx.values())
    print(f'Scenes needing recovery: {n_scenes}  (across {total_missing_rgb} missing files)')

    if n_scenes == 0:
        print('Nothing to recover.')
    else:
        RECOVERY_TMP = os.path.join(TMP_DIR, 'rgb_recovery')
        os.makedirs(RECOVERY_TMP, exist_ok=True)

        # multiprocessing.Pool: each worker has its own GIL. SensorData's
        # constructor is CPU-bound (Python bytecode parses ~150 MB of
        # compressed bytes per scene). ThreadPoolExecutor was serializing
        # this on the GIL — switching to processes gives ~Nx speedup.
        # Match cell 22's pattern (also Pool-based).
        RECOVERY_WORKERS = NETWORK_WORKERS  # cell 6: capped 8 for TUM rate-limiting

        tasks = [
            (sid, scene_to_dir[sid], sorted(scene_to_idx[sid]),
             RECOVERY_TMP, TOOLS_DIR, TARGET_SIZE)
            for sid in sorted(scene_to_idx.keys())
        ]

        print(f'Recovering with {RECOVERY_WORKERS} processes...')
        total_written = 0
        failed = []
        partial = []
        with multiprocessing.Pool(RECOVERY_WORKERS) as pool:
            for result in tqdm(
                pool.imap_unordered(_recover_rgb_for_scene, tasks),
                total=len(tasks), desc='Recovering RGB',
            ):
                sid, status, n_w, n_exp = result
                total_written += n_w
                if status == 'download_failed':
                    failed.append(sid)
                elif n_w < n_exp:
                    partial.append((sid, n_w, n_exp))

        # #38: persist per-scene recovery results to a log.
        recovery_log_path = os.path.join(BASE_OUT_DIR, 'recovery_log.jsonl')
        with open(recovery_log_path, 'w') as _f:
            for sid in sorted(scene_to_idx):
                _f.write(json.dumps({
                    'scene_id': sid,
                    'expected_indices': sorted(scene_to_idx[sid]),
                }) + '\n')

        print(f'\nRecovery complete:')
        print(f'  RGB files written: {total_written} / {n_missing}')
        print(f'  Recovery log: {recovery_log_path}')
        if total_written > 0:
            CHANGES_MADE = True
        if failed:
            print(f'  Download failures: {len(failed)} scenes')
            for s in failed[:10]:
                print(f'    {s}')
        if partial:
            print(f'  Partial recoveries: {len(partial)} scenes')
            for s, w, e in partial[:10]:
                print(f'    {s}: {w}/{e}')

        shutil.rmtree(RECOVERY_TMP, ignore_errors=True)
else:
    print('RUN_RGB_RECOVERY=False -> skipping. Set True only if _rgb.pt files were lost but _hha.pt survived.')

## 11c. CHECKPOINT — Snapshot After RGB Recovery

If you ran the recovery cell above, snapshot the result to Drive so the
recovered `_rgb.pt` files aren't lost on the next runtime crash.

Default ON. Skipped automatically if recovery wasn't actually run.

In [ ]:
SNAPSHOT_AFTER_RECOVERY = True

if SNAPSHOT_AFTER_RECOVERY and 'CHANGES_MADE' in dir() and CHANGES_MADE:
    train_dir = os.path.join(BASE_OUT_DIR, 'train')
    if not os.path.isdir(train_dir) or not any(os.scandir(train_dir)):
        print(f'No data at {train_dir} -> nothing to snapshot.')
    else:
        ensure_pigz()
        snapshot_name = f'{HHA_TARBALL_NAME.replace(".tar.gz", "")}.snapshot.tar.gz'
        snapshot_local = f'/content/{snapshot_name}'

        print(f'Re-snapshotting {BASE_OUT_DIR} (post-recovery)...')
        subprocess.run(
            ['tar', '-I', f'pigz -p {CPU_WORKERS}', '-cf', snapshot_local,
             '-C', os.path.dirname(BASE_OUT_DIR), os.path.basename(BASE_OUT_DIR)],
            check=True,
        )
        upload_to_drive_sync(snapshot_local, 'datasets', snapshot_name)
        os.remove(snapshot_local)
        # HIGH-R4 #6: snapshot now reflects post-recovery state; re-upload
        # the CANONICAL tarball too so downstream notebooks (which read
        # HHA_TARBALL_NAME, not the snapshot) see the recovered data.
        # Without this, canonical is stale until cell 59 runs.
        try:
            ensure_pigz()
            print(f'Re-uploading canonical {HHA_TARBALL_NAME} after recovery...')
            _can_local = f'/content/{HHA_TARBALL_NAME}'
            subprocess.run(
                ['tar', '-I', f'pigz -p {CPU_WORKERS}', '-cf', _can_local,
                 '--exclude=*.failed', '--exclude=*.tmp', '--exclude=progress.json',
                 '-C', os.path.dirname(BASE_OUT_DIR), os.path.basename(BASE_OUT_DIR)],
                check=True,
            )
            upload_to_drive_sync(_can_local, 'datasets', HHA_TARBALL_NAME)
            os.remove(_can_local)
            print('Canonical refreshed.')
        except Exception as _ce:
            print(f'WARNING: canonical refresh after recovery failed: {_ce}. '
                  f'Cell 59 will produce a fresh canonical at end of pipeline.')
elif not SNAPSHOT_AFTER_RECOVERY:
    print('SNAPSHOT_AFTER_RECOVERY=False -> skipping.')
elif 'CHANGES_MADE' not in dir():
    print('CHANGES_MADE flag not set (cells 37/39 did not run) -> nothing to snapshot.')
elif 'RUN_RGB_RECOVERY' in dir() and not RUN_RGB_RECOVERY:
    print('RUN_RGB_RECOVERY=False and no other changes detected -> nothing new to snapshot.')
else:
    # CHANGES_MADE is False but recovery cell ran. Either 0 files needed
    # recovery (clean dataset) or 0 files were successfully recovered.
    print('Recovery ran but no files changed (clean dataset or 0 recoveries) -> nothing new to snapshot.')

## 12. Data Leakage Test

Verify no scene appears in both train and val splits.

In [13]:
# Extract scene IDs from filenames in each split
split_scenes = {}
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    scenes = set()
    if not os.path.exists(split_dir):
        split_scenes[split] = scenes
        continue
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if f.endswith('_rgb.pt'):
                # Extract scene_id: scene0000_00_f00123_rgb.pt -> scene0000_00
                match = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt', f)
                if match:
                    scenes.add(match.group(1))
    split_scenes[split] = scenes

overlap = split_scenes.get('train', set()) & split_scenes.get('val', set())

print(f'Train scenes: {len(split_scenes.get("train", set()))}')
print(f'Val scenes:   {len(split_scenes.get("val", set()))}')

if overlap:
    print(f'\nDATA LEAKAGE DETECTED! {len(overlap)} scenes in both splits:')
    for s in sorted(overlap)[:20]:
        print(f'  {s}')
    raise ValueError(f'Data leakage: {len(overlap)} scenes in both splits')
else:
    print('No data leakage detected.')

Train scenes: 1171
Val scenes:   307
No data leakage detected.


## 13. Frame Diversity Verification

For a sample of scenes, compute pixel-difference between consecutive
sampled frames. This is a heuristic sanity check, not a guarantee --
uniform walls may flag as duplicates. The even-spacing strategy is the
primary guarantee of diversity.

In [ ]:
random.seed(42)
DIVERSITY_SAMPLE = 10  # Number of scenes to check

all_train_scenes = sorted(split_scenes.get('train', set()))
sample_scenes = random.sample(
    all_train_scenes, min(DIVERSITY_SAMPLE, len(all_train_scenes))
)
sample_scenes_set = set(sample_scenes)

# Build scene_id -> sorted [paths] index in one pass (vs per-scene dir walks).
scene_to_paths = {}
train_root = os.path.join(BASE_OUT_DIR, 'train')
if not os.path.isdir(train_root):
    print(f'  WARNING: {train_root} does not exist, skipping diversity check')
    sample_scenes = []  # short-circuit the loop below
for cls_entry in (os.scandir(train_root) if os.path.isdir(train_root) else []):
    if not cls_entry.is_dir():
        continue
    for fname in os.listdir(cls_entry.path):
        if not fname.endswith('_rgb.pt'):
            continue
        m = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt$', fname)
        if not m:
            continue
        sid = m.group(1)
        if sid in sample_scenes_set:  # O(1) set lookup
            scene_to_paths.setdefault(sid, []).append(os.path.join(cls_entry.path, fname))
for sid in scene_to_paths:
    scene_to_paths[sid].sort()  # consistent ordering for consecutive-frame diff

print('Frame diversity check (mean absolute pixel difference between consecutive frames):')
print(f'Checking {len(sample_scenes)} scenes...\n')

for scene_id in sample_scenes:
    scene_files = scene_to_paths.get(scene_id, [])
    if len(scene_files) < 2:
        print(f'  {scene_id}: only {len(scene_files)} frame(s), skipping')
        continue

    diffs = []
    prev = torch.load(scene_files[0], weights_only=True).float()
    for fpath in scene_files[1:]:
        curr = torch.load(fpath, weights_only=True).float()
        diff = (curr - prev).abs().mean().item()
        diffs.append(diff)
        prev = curr

    mean_diff = np.mean(diffs)
    min_diff = np.min(diffs)
    print(f'  {scene_id}: {len(scene_files)} frames, '
          f'mean_diff={mean_diff:.1f}, min_diff={min_diff:.1f}')
    if min_diff < 5.0:
        print(f'    WARNING: Very low min diff ({min_diff:.1f}) -- '
              f'possible near-duplicate frames')

## 14. Dataset Statistics

Per-class distribution (train & val), frames per scene histogram,
depth range stats, total sample counts.

In [ ]:
# Per-class frame counts
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        continue
    print(f'\n=== {split.upper()} ===')
    total = 0
    class_counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        n = len(glob.glob(os.path.join(cls_dir, '*_rgb.pt')))
        class_counts[cls] = n
        total += n
    for cls, n in sorted(class_counts.items()):
        pct = 100.0 * n / total if total > 0 else 0
        print(f'  {cls:30s} {n:5d}  ({pct:5.1f}%)')
    print(f'  {"TOTAL":30s} {total:5d}')

# Frames per scene histogram
scene_frame_counts = Counter()
for split in ['train', 'val']:
    split_dir = os.path.join(BASE_OUT_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if f.endswith('_rgb.pt'):
                match = re.match(r'(scene\d+_\d+)_f\d+_rgb\.pt', f)
                if match:
                    scene_frame_counts[match.group(1)] += 1

frame_counts = list(scene_frame_counts.values())
if frame_counts:
    print(f'\n=== Frames per scene ===')
    print(f'  Scenes: {len(frame_counts)}')
    print(f'  Min:    {min(frame_counts)}')
    print(f'  Max:    {max(frame_counts)}')
    print(f'  Mean:   {np.mean(frame_counts):.1f}')
    print(f'  Median: {np.median(frame_counts):.1f}')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(frame_counts, bins=20, edgecolor='black')
    ax.set_xlabel('Frames per scene')
    ax.set_ylabel('Count')
    ax.set_title('Frame count distribution')
    plt.tight_layout()
    plt.show()
    plt.close(fig)  # FIX #13: prevent figure accumulation on cell re-run

# Depth range stats (sample) -- only meaningful when raw depth tensors exist.
if HHA_MODE != 'hha-only':
    print(f'\n=== Depth range stats (sample of 100 frames) ===')
    depth_files = sorted(glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_depth.pt')))
    # Local seeded RNG — was random.shuffle on the global module, which is
    # mutated by cell 45's random.seed(42) and creates a hidden execution-order
    # dependency. Sort first so glob's filesystem-order doesn't matter either.
    sample_depth = random.Random(42).sample(depth_files, min(100, len(depth_files)))
    if not sample_depth:
        print('  WARNING: No _depth.pt files found. Skipping range stats.')
    else:
        depth_mins = []
        depth_maxs = []
        zero_fracs = []
        for dp in sample_depth:
            d = torch.load(dp, weights_only=True).float()
            if (d > 0).any():
                depth_mins.append(d[d > 0].min().item())
            depth_maxs.append(d.max().item())
            zero_fracs.append((d == 0).float().mean().item())
        if depth_mins:
            print(f'  Min depth (non-zero, mm): {np.min(depth_mins):.0f}')
        print(f'  Max depth (mm):           {np.max(depth_maxs):.0f}')
        print(f'  Mean zero fraction:       {np.mean(zero_fracs):.3f}')
else:
    print(f'\n=== Depth range stats: SKIPPED (HHA_MODE=hha-only, no _depth.pt files) ===')

# HHA range stats (sample) — quick smoke test before the heavy norm_stats cell.
if HHA_MODE != 'none':
    print(f'\n=== HHA channel preview (sample of 100 frames) ===')
    hha_files = glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_hha.pt'))
    _rng = random.Random(42)
    sample_hha = _rng.sample(hha_files, min(100, len(hha_files)))
    if not sample_hha:
        print('  WARNING: no _hha.pt files found.')
    else:
        per_channel_min = [float('inf')] * 3
        per_channel_max = [float('-inf')] * 3
        per_channel_finite = [0, 0, 0]
        per_channel_total = [0, 0, 0]
        for hp in sample_hha:
            t = torch.load(hp, weights_only=True).float()
            for c in range(3):
                ch = t[c]
                mask = torch.isfinite(ch)
                vals = ch[mask]
                per_channel_finite[c] += int(mask.sum())
                per_channel_total[c]  += int(ch.numel())
                if vals.numel():
                    per_channel_min[c] = min(per_channel_min[c], float(vals.min()))
                    per_channel_max[c] = max(per_channel_max[c], float(vals.max()))
        for c, lab in enumerate(['disparity (1/m)', 'height (m)', 'angle (deg)']):
            nan_rate = 1.0 - per_channel_finite[c] / max(per_channel_total[c], 1)
            print(f'  ch{c} ({lab:18s}) min={per_channel_min[c]:8.3f} '
                  f'max={per_channel_max[c]:8.3f}  nan_rate={nan_rate:.3f}')

## 15. Normalization Stats

Streaming (Welford) computation of RGB mean/std and depth mean/std
from **train split only**. Depth stats exclude pixels with value 0
(sentinel for missing data). Depth values are converted to meters
(`/ 1000.0`) before stats computation.

In [ ]:
# =====================================================================
# Normalization stats — ThreadPoolExecutor variant.
#
# Why threads, not multiprocessing.Pool: forking workers in Jupyter AFTER
# matplotlib/torch state has been touched in the parent kernel sometimes
# stalls indefinitely (worker startup hangs). ThreadPoolExecutor shares
# kernel state cleanly. torch.load releases the GIL during the C++ pickle
# read, and numpy reductions release the GIL too — so threads parallelize
# the I/O-bound and vectorized portions even though pure-Python parts
# serialize on the GIL.
# =====================================================================

SAMPLE_N_PER_FILE = 100
COMPUTE_DEPTH_STATS = (HHA_MODE != 'hha-only')

# Skip recompute if dataset is unchanged from Drive AND norm_stats.json on
# disk already has the required keys.
SKIP_NORM_STATS = False
if 'CHANGES_MADE' in dir() and not CHANGES_MADE and 'DATA_AVAILABLE' in dir() and DATA_AVAILABLE:
    _ns_path = os.path.join(BASE_OUT_DIR, 'norm_stats.json')
    if os.path.exists(_ns_path) and os.path.getsize(_ns_path) > 0:
        with open(_ns_path) as _f:
            _ns_existing = json.load(_f)
        _required = ['rgb_mean', 'rgb_std']
        if HHA_MODE != 'none':
            # R5 #35: cell 53 reads norm_stats['hha_ranges'][c]['p1' etc.]; an
            # older norm_stats.json from a pre-hha_ranges run lacks this and
            # crashes cell 53. Force recompute so the new fields are written.
            _required += ['hha_mean', 'hha_std', 'hha_ranges']
        if all(k in _ns_existing for k in _required):
            SKIP_NORM_STATS = True
            norm_stats = _ns_existing
            print(f'SKIPPED: norm_stats unchanged (dataset unchanged + norm_stats.json valid).')
            print(f'  Loaded from {_ns_path}')

if not SKIP_NORM_STATS:


    def _stats_worker(args):
        """Process one frame. Returns dict of partial stats."""
        rgb_path, depth_path, hha_path, sample_n = args
        out = {'rgb': None, 'depth': None, 'hha': None}

        rgb = torch.load(rgb_path, weights_only=True).numpy()  # uint8 [3,H,W]
        rgb_d = rgb.astype(np.float64)
        out['rgb'] = (
            np.array([rgb.shape[1] * rgb.shape[2]] * 3, dtype=np.int64),
            rgb_d.sum(axis=(1, 2)),
            (rgb_d ** 2).sum(axis=(1, 2)),
        )

        if depth_path is not None and os.path.exists(depth_path):
            depth = torch.load(depth_path, weights_only=True).numpy().astype(np.float64).ravel()
            valid = depth[depth > 0] / 1000.0
            if len(valid) > 0:
                out['depth'] = (int(len(valid)), float(valid.sum()), float((valid ** 2).sum()))

        if hha_path is not None and os.path.exists(hha_path):
            hha = torch.load(hha_path, weights_only=True).to(torch.float32).numpy()
            H, W = hha.shape[1], hha.shape[2]
            ns = np.zeros(3, dtype=np.int64)
            sums = np.zeros(3, dtype=np.float64)
            sumsqs = np.zeros(3, dtype=np.float64)
            mins = np.full(3, np.inf, dtype=np.float64)
            maxs = np.full(3, -np.inf, dtype=np.float64)
            samples = [None, None, None]
            # Deterministic per-file seed (path-derived) — sampled pixels and
            # therefore percentile estimates reproduce across runs.
            import hashlib as _h
            _seed = int(_h.sha1(hha_path.encode()).hexdigest()[:8], 16)
            _rng = np.random.default_rng(_seed)
            for c in range(3):
                ch = hha[c].ravel()
                mask = np.isfinite(ch)
                vals = ch[mask]
                if vals.size == 0:
                    continue
                vals_d = vals.astype(np.float64)
                ns[c] = vals.size
                sums[c] = vals_d.sum()
                sumsqs[c] = (vals_d ** 2).sum()
                mins[c] = float(vals.min())
                maxs[c] = float(vals.max())
                if vals.size <= sample_n:
                    samples[c] = vals.copy()
                else:
                    idx = _rng.choice(vals.size, sample_n, replace=False)
                    samples[c] = vals[idx].copy()
            out['hha'] = (ns, sums, sumsqs, mins, maxs, samples, H * W)

        return out


    print('Scanning per-sample files from train split...')
    all_rgb_paths = sorted(glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_rgb.pt')))
    print(f'Train samples: {len(all_rgb_paths)}')

    if COMPUTE_DEPTH_STATS:
        all_depth_paths = [p.replace('_rgb.pt', '_depth.pt') for p in all_rgb_paths]
    else:
        all_depth_paths = [None] * len(all_rgb_paths)

    if HHA_MODE != 'none':
        all_hha_paths = [p.replace('_rgb.pt', '_hha.pt') for p in all_rgb_paths]
    else:
        all_hha_paths = [None] * len(all_rgb_paths)

    tasks = list(zip(all_rgb_paths, all_depth_paths, all_hha_paths,
                     [SAMPLE_N_PER_FILE] * len(all_rgb_paths)))

    if not tasks:
        raise RuntimeError(
            f'No _rgb.pt files found under {BASE_OUT_DIR}/train. '
            f'Cell 22 did not produce any data (or it was wiped). '
            f'Cannot compute normalization stats.'
        )

    # --- SANITY CHECK: 10 RANDOM files (not alphabetically first) ---
    # Avoids being blind to class-specific corruption — apartment is always
    # first if we just slice tasks[:10].
    import random as _random
    _sanity_rng = _random.Random(42)
    _n_sanity = min(10, len(tasks))
    _sanity_idx = _sanity_rng.sample(range(len(tasks)), _n_sanity)
    _sanity_idx_set = set(_sanity_idx)
    print(f'Sanity check: processing {_n_sanity} random files inline...')
    _t0 = time.time()
    _sanity_results = []
    _sanity_errors = []
    for _i in _sanity_idx:
        _t = tasks[_i]
        try:
            _sanity_results.append(_stats_worker(_t))
        except Exception as _e:
            _sanity_errors.append((_t[0], type(_e).__name__, str(_e)[:200]))
    if not _sanity_results:
        raise RuntimeError(
            f'All sanity-check files failed (random sample of {_n_sanity}). '
            f'Errors: {_sanity_errors[:3]}'
        )
    if _sanity_errors:
        print(f'  {len(_sanity_errors)}/{_n_sanity} sanity files errored — proceeding:')
        for _p, _t, _m in _sanity_errors[:3]:
            print(f'    {_p}: {_t}: {_m}')
    _test = _sanity_results[0]
    _dt = (time.time() - _t0) / max(len(_sanity_results), 1)
    print(f'  Per-file: ~{_dt*1000:.0f} ms')
    # ETA fudge factor 1.7x — GIL contention + I/O make perfect linear scaling rare.
    _eta_min = (_dt * len(tasks) * 1.7 / max(MAX_WORKERS, 1)) / 60
    print(f'  Projected wall time with {MAX_WORKERS} threads: ~{_eta_min:.1f} min (incl. 1.7x fudge)')

    # Aggregators
    rgb_n     = np.zeros(3, dtype=np.int64)
    rgb_sum   = np.zeros(3, dtype=np.float64)
    rgb_sumsq = np.zeros(3, dtype=np.float64)
    depth_n_total      = 0
    depth_sum_total    = 0.0
    depth_sumsq_total  = 0.0
    hha_n     = np.zeros(3, dtype=np.int64)
    hha_sum   = np.zeros(3, dtype=np.float64)
    hha_sumsq = np.zeros(3, dtype=np.float64)
    hha_min   = np.full(3, np.inf,  dtype=np.float64)
    hha_max   = np.full(3, -np.inf, dtype=np.float64)
    hha_samples = [[], [], []]
    hha_total_pixels = 0


    def _aggregate(result):
        """Merge one worker's partial stats into the global accumulators."""
        global rgb_n, rgb_sum, rgb_sumsq
        global depth_n_total, depth_sum_total, depth_sumsq_total
        global hha_n, hha_sum, hha_sumsq
        global hha_min, hha_max, hha_total_pixels
        n, s, ss = result['rgb']
        rgb_n     += n
        rgb_sum   += s
        rgb_sumsq += ss
        if result['depth'] is not None:
            n, s, ss = result['depth']
            depth_n_total     += n
            depth_sum_total   += s
            depth_sumsq_total += ss
        if result['hha'] is not None:
            ns, sums, sumsqs, mins, maxs, samples, hw = result['hha']
            hha_n     += ns
            hha_sum   += sums
            hha_sumsq += sumsqs
            hha_min = np.minimum(hha_min, mins)
            hha_max = np.maximum(hha_max, maxs)
            for c in range(3):
                if samples[c] is not None:
                    hha_samples[c].append(samples[c])
            hha_total_pixels += hw


    # Aggregate ALL sanity results so they aren't reprocessed by the pool.
    for _r in _sanity_results:
        _aggregate(_r)

    # Build the remaining task list excluding the random sanity indices.
    _remaining = [t for i, t in enumerate(tasks) if i not in _sanity_idx_set]
    print(f'Computing stats with {MAX_WORKERS} threads (skipping {len(_sanity_idx_set)} sanity tasks)...')
    worker_errors = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_stats_worker, t) for t in _remaining]
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Computing stats'):
            try:
                result = fut.result()
            except Exception as e:
                worker_errors.append((type(e).__name__, str(e)[:200]))
                continue
            _aggregate(result)

    # Aggregate worker errors so mass-failure is visible (not just a fast progress bar).
    if worker_errors:
        _pool_n = len(tasks) - len(_sanity_idx_set)
        print(f'\n  WORKER ERRORS: {len(worker_errors)} / {_pool_n} '
              f'({100*len(worker_errors)/_pool_n:.1f}%)')
        from collections import Counter as _Counter
        _by_type = _Counter(t for t, _ in worker_errors)
        for _type, _n in _by_type.most_common(5):
            print(f'    {_type}: {_n}')
        if len(worker_errors) > 0.05 * _pool_n:
            raise RuntimeError(
                f'Norm-stats worker error rate too high ({len(worker_errors)} failures). '
                f'Stats unreliable — inspect errors and fix before retrying.'
            )

    # --- Finalize RGB stats (Bessel-corrected unbiased variance) ---
    rgb_mean   = rgb_sum / rgb_n
    rgb_var    = np.clip((rgb_sumsq - (rgb_sum ** 2) / rgb_n) / np.maximum(rgb_n - 1, 1), 0.0, None)
    rgb_std    = np.sqrt(rgb_var)
    rgb_mean_01 = rgb_mean / 255.0
    rgb_std_01  = rgb_std  / 255.0

    norm_stats = {
        'rgb_mean': rgb_mean_01.tolist(),
        'rgb_std':  rgb_std_01.tolist(),
    }
    print(f'\nRGB mean: {rgb_mean_01}')
    print(f'RGB std:  {rgb_std_01}')
    print(f'RGB pixels per channel: {rgb_n}')

    # --- Finalize depth stats ---
    if COMPUTE_DEPTH_STATS:
        if depth_n_total > 1:
            depth_mean = depth_sum_total / depth_n_total
            depth_var = max(0.0, (depth_sumsq_total - depth_n_total * depth_mean ** 2) / max(depth_n_total - 1, 1))
            depth_std = float(depth_var ** 0.5)
        else:
            depth_mean = 0.0
            depth_std = 0.0
        norm_stats['depth_mean'] = [float(depth_mean)]
        norm_stats['depth_std']  = [float(depth_std)]
        print(f'Depth mean: {depth_mean:.4f} m')
        print(f'Depth std:  {depth_std:.4f} m')
        print(f'Depth pixels (non-zero): {depth_n_total:,}')
    else:
        print('Depth Welford: SKIPPED (HHA_MODE=hha-only).')

    # --- Finalize HHA stats ---
    if HHA_MODE != 'none':
        print('\n=== HHA channel stats ===')
        for c in range(3):
            if hha_n[c] == 0:
                raise RuntimeError(f'HHA channel {c} has no finite values')

        hha_mean_arr = hha_sum / hha_n
        hha_var = np.clip((hha_sumsq - hha_n * hha_mean_arr ** 2) / np.maximum(hha_n - 1, 1), 0.0, None)
        hha_std_arr = np.sqrt(hha_var)

        hha_ranges = []
        for c in range(3):
            samples = np.concatenate(hha_samples[c]) if hha_samples[c] else np.array([])
            if samples.size:
                p1   = float(np.percentile(samples, 1))
                p99  = float(np.percentile(samples, 99))
                p999 = float(np.percentile(samples, 99.9))
            else:
                p1 = p99 = p999 = float('nan')
            nan_rate = float(1.0 - hha_n[c] / max(hha_total_pixels, 1))
            hha_ranges.append({
                'min': float(hha_min[c]),
                'max': float(hha_max[c]),
                'p1': p1, 'p99': p99, 'p99_9': p999,
                'nan_rate': nan_rate,
            })

        norm_stats['hha_mean']   = hha_mean_arr.tolist()
        norm_stats['hha_std']    = hha_std_arr.tolist()
        norm_stats['hha_ranges'] = hha_ranges
        print(f'HHA mean: {hha_mean_arr.tolist()}')
        print(f'HHA std:  {hha_std_arr.tolist()}')
        for c, lab in enumerate(['disparity', 'height_m', 'angle_deg']):
            r = hha_ranges[c]
            print(f'  {lab}: min={r["min"]:.3f}, max={r["max"]:.3f}, '
                  f'p1={r["p1"]:.3f}, p99={r["p99"]:.3f}, p99.9={r["p99_9"]:.3f}, '
                  f'nan_rate={r["nan_rate"]:.4f}')

        with open(os.path.join(BASE_OUT_DIR, 'norm_stats.json'), 'w') as f:
            json.dump(norm_stats, f, indent=2)
        print(f'Updated norm_stats.json with HHA keys')

## 16. Write Metadata Files

In [ ]:
# Resilience: if norm_stats isn't in memory (e.g., user re-runs this cell
# after a kernel restart), load from disk. Cell 45 always writes it before this.
if 'norm_stats' not in dir():
    norm_stats_path_in = os.path.join(BASE_OUT_DIR, 'norm_stats.json')
    if not os.path.isfile(norm_stats_path_in):
        raise RuntimeError(
            f'norm_stats not in memory and not on disk at {norm_stats_path_in}. '
            f'Run cell 45 first.'
        )
    with open(norm_stats_path_in) as _f:
        norm_stats = json.load(_f)
    print(f'Loaded norm_stats from disk: {norm_stats_path_in}')

# class_names.txt
# Build per-class counts first so we can warn about empty classes
# (still writing all 20 names so the label index is stable; this is a sanity print).
_per_split_counts = {}
for _sp in ('train', 'val'):
    _split_dir = os.path.join(BASE_OUT_DIR, _sp)
    if not os.path.isdir(_split_dir):
        continue
    _per_split_counts[_sp] = {}
    for cls in SCANNET_SCENE_TYPES:
        _cls_dir = os.path.join(_split_dir, cls)
        n = (len(glob.glob(os.path.join(_cls_dir, '*_rgb.pt')))
             if os.path.isdir(_cls_dir) else 0)
        _per_split_counts[_sp][cls] = n
        if n == 0:
            print(f'  WARNING: class {cls!r} has 0 frames in {_sp}/. '
                  f'Per-class metrics on this class will be undefined.')

class_names_path = os.path.join(BASE_OUT_DIR, 'class_names.txt')
with open(class_names_path, 'w') as f:
    for name in SCANNET_SCENE_TYPES:
        f.write(f'{name}\n')
print(f'Wrote {class_names_path} ({len(SCANNET_SCENE_TYPES)} classes)')

# norm_stats.json
norm_stats_path = os.path.join(BASE_OUT_DIR, 'norm_stats.json')
with open(norm_stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)
print(f'Wrote {norm_stats_path}')

# dataset_info.txt
train_count = len(glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_rgb.pt')))
val_count   = len(glob.glob(os.path.join(BASE_OUT_DIR, 'val',   '*', '*_rgb.pt')))

info_path = os.path.join(BASE_OUT_DIR, 'dataset_info.txt')
with open(info_path, 'w') as f:
    f.write(f'ScanNet Pretrain Dataset ({TARGET_SIZE}x{TARGET_SIZE})\n')
    f.write(f'Created: {time.strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'HHA mode: {HHA_MODE}\n')
    f.write(f'Num classes: {len(SCANNET_SCENE_TYPES)}\n')
    f.write(f'Train samples: {train_count}\n')
    f.write(f'Val samples: {val_count}\n')
    f.write(f'Total samples: {train_count + val_count}\n')
    # Actual per-scene distribution (varies with blur/pose drops).
    _scene_counts = Counter()
    for _sp in ('train', 'val'):
        for _f in glob.glob(os.path.join(BASE_OUT_DIR, _sp, '*', '*_rgb.pt')):
            _m = re.match(r'(scene\d+_\d+)_f', os.path.basename(_f))
            if _m: _scene_counts[_m.group(1)] += 1
    if _scene_counts:
        _vals = sorted(_scene_counts.values())
        f.write(
            f'Frames per scene (target {FRAMES_PER_SCENE}): '
            # FIX #14: was _vals[len(_vals)//2] which is upper-middle (off-by-1
            # on even-length lists). np.median averages the two middle values.
            f'min={_vals[0]} median={float(np.median(_vals)):.1f} max={_vals[-1]} '
            f'mean={sum(_vals)/len(_vals):.1f}\n'
        )
    else:
        f.write(f'Frames per scene: {FRAMES_PER_SCENE} (target; no files found)\n')
    f.write(f'Target size: {TARGET_SIZE}\n')
    f.write(f'RGB format: uint8 [3, {TARGET_SIZE}, {TARGET_SIZE}]\n')
    if HHA_MODE != 'hha-only':
        f.write(f'Depth format: uint16 [1, {TARGET_SIZE}, {TARGET_SIZE}] (mm)\n')
        f.write(f'Depth stats: exclude zero sentinels, meters (/1000.0)\n')
    if HHA_MODE != 'none':
        f.write(f'HHA format: {HHA_DTYPE} [3, {TARGET_SIZE}, {TARGET_SIZE}]\n')
        f.write(f'HHA channels: disparity (1/m), height (m, world frame), angle (deg with gravity)\n')
    # #15: split provenance.
    # HIGH #6: REBALANCE_MOVED isn't set when running from a restored dataset
    # (cell 35 didn't run this session). Read the persisted value from
    # rebalance.json so the info file stays accurate across restores.
    _moved = globals().get('REBALANCE_MOVED')
    if _moved is None:
        _rb_path = os.path.join(BASE_OUT_DIR, 'rebalance.json')
        if os.path.isfile(_rb_path):
            try:
                with open(_rb_path) as _rbf:
                    _moved = int(json.load(_rbf).get('rebalance_moved', 0))
            except (OSError, ValueError):
                _moved = 'unknown'
        else:
            _moved = 'unknown'
    f.write(f'\nSplit:\n')
    if _moved and _moved != 'unknown' and _moved > 0:
        f.write(f'  NOT canonical ScanNet split — {_moved} scenes moved val->train '
                f'to hit ~90/10 frame ratio. Do not compare to canonical-split benchmarks.\n')
    else:
        f.write(f'  Canonical ScanNet train/val split (no rebalance moves).\n')
    f.write(f'\nNorm stats:\n')
    for k, v in norm_stats.items():
        f.write(f'  {k}: {v}\n')
    f.write(f'\nClass names:\n')
    for i, name in enumerate(SCANNET_SCENE_TYPES):
        f.write(f'  {i}: {name}\n')
print(f'Wrote {info_path}')

# #24: also write dataset_info.json (programmatic counterpart to dataset_info.txt).
info_json_path = os.path.join(BASE_OUT_DIR, 'dataset_info.json')
_info_json = {
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'hha_mode': HHA_MODE,
    'hha_dtype': HHA_DTYPE if HHA_MODE != 'none' else None,
    'num_classes': len(SCANNET_SCENE_TYPES),
    'class_names': list(SCANNET_SCENE_TYPES),
    'train_samples': train_count,
    'val_samples': val_count,
    'total_samples': train_count + val_count,
    'frames_per_scene_target': FRAMES_PER_SCENE,
    'target_size': TARGET_SIZE,
    'rgb_format': f'uint8 [3, {TARGET_SIZE}, {TARGET_SIZE}]',
    'hha_format': (f'{HHA_DTYPE} [3, {TARGET_SIZE}, {TARGET_SIZE}]'
                   if HHA_MODE != 'none' else None),
    'depth_format': (f'uint16 [1, {TARGET_SIZE}, {TARGET_SIZE}] (mm)'
                     if HHA_MODE != 'hha-only' else None),
    'norm_stats': norm_stats,
}
with open(info_json_path, 'w') as f:
    json.dump(_info_json, f, indent=2)
print(f'Wrote {info_json_path}')

## 17. ScanNet vs SUN RGB-D Comparison

Validates that ScanNet is suitable pretraining data for a model that will
fine-tune on SUN RGB-D. Compares normalization statistics, depth distributions,
missing-data ratios, and class overlap.

**Requires:** `sunrgbd_19_traintest.tar.gz` on Drive (copies only `norm_stats.json`,
`class_names.txt`, and a sample of tensors -- does NOT extract the full dataset).

In [ ]:
# =========================================================================
# ScanNet vs SUN RGB-D — Pretraining Fitness Check
# =========================================================================
# In HHA mode, compares ScanNet HHA stats against SUN HHA stats (apples-to-apples).
# In raw-depth mode, compares raw depth distributions.
# =========================================================================

if HHA_MODE != 'none':
    SUN_TAR = '/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz'
    SUN_LABEL = 'SUN HHA'
    SUN_DEPTH_FILE = 'hha_tensors.pt'
else:
    SUN_TAR = '/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz'
    SUN_LABEL = 'SUN raw-depth'
    SUN_DEPTH_FILE = 'depth_tensors.pt'

SUN_TMP = '/content/sun_comparison_tmp'
HAS_SCANNET_DEPTH = ('depth_mean' in norm_stats)
HAS_SCANNET_HHA   = ('hha_mean'   in norm_stats)

if not os.path.exists(SUN_TAR):
    print(f'WARNING: SUN dataset not found at {SUN_TAR}')
    print(f'Skipping comparison. Upload {os.path.basename(SUN_TAR)} to Drive to run this cell.')
else:
    print(f'Comparing ScanNet ({"HHA" if HHA_MODE != "none" else "raw"}) against {SUN_LABEL}.')
    os.makedirs(SUN_TMP, exist_ok=True)

    # Extract metadata + payload tensor.
    tar_cmd = ['tar', 'xzf', SUN_TAR, '-C', SUN_TMP, '--wildcards',
               '*/norm_stats.json', '*/class_names.txt',
               f'*/train/{SUN_DEPTH_FILE}', '*/train/labels.txt']
    _tar_result = subprocess.run(tar_cmd, capture_output=True, text=True)
    if _tar_result.returncode != 0:
        # FIX #12: comparison is secondary to the dataset; degrade gracefully
        # instead of aborting the cell. Lets the operator still proceed to
        # tar+upload (cell 59) without hand-fixing the SUN tarball.
        print(f'WARNING: tar extraction of SUN tarball failed '
              f'(returncode={_tar_result.returncode}).')
        print(f'  stderr: {_tar_result.stderr[:500]}')
        print(f'  -> Skipping SUN comparison. ScanNet outputs unaffected.')
        shutil.rmtree(SUN_TMP, ignore_errors=True)
        # Set a sentinel that downstream blocks below check.
        _SKIP_SUN_COMPARE = True
    else:
        _SKIP_SUN_COMPARE = False

    if not _SKIP_SUN_COMPARE:
        # Find the extracted root.
        sun_root = SUN_TMP
        for root, dirs, files in os.walk(SUN_TMP):
            if 'norm_stats.json' in files:
                sun_root = root
                break

        with open(os.path.join(sun_root, 'norm_stats.json')) as f:
            sun_stats = json.load(f)
        with open(os.path.join(sun_root, 'class_names.txt')) as f:
            sun_classes = [l.strip().split(': ', 1)[-1] for l in f if l.strip()]

        SUN_HAS_HHA = ('hha_mean' in sun_stats)

        # =====================================================================
        # 1. RGB Normalization Statistics (always comparable)
        # =====================================================================
        print()
        print('=' * 70)
        print('1. RGB NORMALIZATION STATISTICS')
        print('=' * 70)
        print(f'{"":30s} {"ScanNet":>15s} {SUN_LABEL:>15s} {"Diff":>10s}')
        print('-' * 70)
        for ch, name in enumerate(['R', 'G', 'B']):
            sc = norm_stats['rgb_mean'][ch]
            su = sun_stats['rgb_mean'][ch]
            print(f'  RGB mean ({name})               {sc:15.4f} {su:15.4f} {abs(sc-su):10.4f}')
        for ch, name in enumerate(['R', 'G', 'B']):
            sc = norm_stats['rgb_std'][ch]
            su = sun_stats['rgb_std'][ch]
            print(f'  RGB std  ({name})               {sc:15.4f} {su:15.4f} {abs(sc-su):10.4f}')

        rgb_mean_diff = max(abs(norm_stats['rgb_mean'][i] - sun_stats['rgb_mean'][i]) for i in range(3))
        rgb_std_diff  = max(abs(norm_stats['rgb_std'][i]  - sun_stats['rgb_std'][i])  for i in range(3))
        print()
        if rgb_mean_diff < 0.05:
            print('  RGB mean: GOOD (< 0.05 divergence)')
        elif rgb_mean_diff < 0.10:
            print(f'  RGB mean: ACCEPTABLE ({rgb_mean_diff:.3f} divergence)')
        else:
            print(f'  RGB mean: WARNING — large divergence ({rgb_mean_diff:.3f})')
        if rgb_std_diff < 0.05:
            print('  RGB std:  GOOD (< 0.05 divergence)')
        elif rgb_std_diff < 0.10:
            print(f'  RGB std:  ACCEPTABLE ({rgb_std_diff:.3f} divergence)')
        else:
            print(f'  RGB std:  WARNING — large divergence ({rgb_std_diff:.3f})')

        # =====================================================================
        # 2. Depth or HHA Distribution Comparison
        # =====================================================================
        print()
        print('=' * 70)
        if HAS_SCANNET_HHA and SUN_HAS_HHA:
            print('2. HHA CHANNEL STATISTICS (ScanNet HHA vs SUN HHA)')
        elif HAS_SCANNET_DEPTH:
            print('2. DEPTH STATISTICS (raw depth in meters)')
        else:
            print('2. DEPTH/HHA STATISTICS — UNAVAILABLE')
        print('=' * 70)

        hha_max_diff_rel = 0.0  # tracked for verdict

        if HAS_SCANNET_HHA and SUN_HAS_HHA:
            # Direct HHA channel comparison (apples-to-apples).
            ch_names = ['disparity (1/m)', 'height (m)    ', 'angle (deg)   ']
            print(f'{"":30s} {"ScanNet":>15s} {"SUN":>15s} {"|Δ|/SUN":>12s}')
            print('-' * 75)
            for c, name in enumerate(ch_names):
                sc_m = norm_stats['hha_mean'][c]
                su_m = sun_stats['hha_mean'][c]
                sc_s = norm_stats['hha_std'][c]
                su_s = sun_stats['hha_std'][c]
                denom_m = max(abs(su_m), 1e-6)
                denom_s = max(abs(su_s), 1e-6)
                rel_m = abs(sc_m - su_m) / denom_m
                rel_s = abs(sc_s - su_s) / denom_s
                hha_max_diff_rel = max(hha_max_diff_rel, rel_m, rel_s)
                print(f'  {name} mean         {sc_m:15.4f} {su_m:15.4f} {rel_m:11.1%}')
                print(f'  {name} std          {sc_s:15.4f} {su_s:15.4f} {rel_s:11.1%}')

            # HHA percentile-range comparison.
            print()
            print('  Per-channel ranges (p1 / p99 / max):')
            for c, name in enumerate(ch_names):
                sr = norm_stats['hha_ranges'][c]
                ur = sun_stats['hha_ranges'][c]
                print(f'    {name}')
                print(f'      ScanNet: p1={sr["p1"]:8.3f}  p99={sr["p99"]:8.3f}  max={sr["max"]:8.3f}  nan_rate={sr["nan_rate"]:.3f}')
                print(f'      SUN:     p1={ur["p1"]:8.3f}  p99={ur["p99"]:8.3f}  max={ur["max"]:8.3f}  nan_rate={ur["nan_rate"]:.3f}')

            print()
            if hha_max_diff_rel < 0.20:
                print(f'  HHA distributions: GOOD (max relative divergence {hha_max_diff_rel:.1%})')
            elif hha_max_diff_rel < 0.50:
                print(f'  HHA distributions: ACCEPTABLE (max relative divergence {hha_max_diff_rel:.1%})')
            else:
                print(f'  HHA distributions: WARNING — large divergence ({hha_max_diff_rel:.1%})')

        elif HAS_SCANNET_DEPTH and not SUN_HAS_HHA:
            # Raw depth comparison (unchanged from old logic).
            sun_depth_in_meters = sun_stats['depth_mean'][0] >= 1.0
            if sun_depth_in_meters:
                sc_dm = norm_stats['depth_mean'][0]; su_dm = sun_stats['depth_mean'][0]
                sc_ds = norm_stats['depth_std'][0];  su_ds = sun_stats['depth_std'][0]
                print(f'  Depth mean (m): ScanNet={sc_dm:.4f}, SUN={su_dm:.4f}, diff={abs(sc_dm-su_dm):.4f}')
                print(f'  Depth std  (m): ScanNet={sc_ds:.4f}, SUN={su_ds:.4f}, diff={abs(sc_ds-su_ds):.4f}')
            else:
                print(f'  Depth units differ between datasets — direct comparison not meaningful.')
                print(f'  ScanNet depth_mean: {norm_stats["depth_mean"][0]:.4f}')
                print(f'  SUN depth_mean:     {sun_stats["depth_mean"][0]:.4f}')

        else:
            print('  No comparable depth/HHA stats available.')
            print(f'    ScanNet has HHA: {HAS_SCANNET_HHA}, has raw depth: {HAS_SCANNET_DEPTH}')
            print(f'    SUN has HHA:     {SUN_HAS_HHA}')
            if HHA_MODE != 'none' and not SUN_HAS_HHA:
                print(f'  Tip: the SUN tarball at {SUN_TAR} appears to be the raw-depth version.')
                print(f'  For a meaningful HHA-vs-HHA comparison, point SUN_TAR at sunrgbd_19_hha.tar.gz.')

        # =====================================================================
        # 3. Class Overlap Analysis
        # =====================================================================
        print()
        print('=' * 70)
        print('3. CLASS OVERLAP ANALYSIS')
        print('=' * 70)
        def _norm_name(s):
            return s.lower().replace(' ', '_').replace('/', '_')
        scannet_classes_lower = {_norm_name(c) for c in SCANNET_SCENE_TYPES}
        sun_classes_lower     = {_norm_name(c) for c in sun_classes}
        overlap = scannet_classes_lower & sun_classes_lower
        scannet_only = scannet_classes_lower - sun_classes_lower
        sun_only = sun_classes_lower - scannet_classes_lower
        print(f'  ScanNet classes:   {len(SCANNET_SCENE_TYPES)}')
        print(f'  SUN RGB-D classes: {len(sun_classes)}')
        print(f'  Exact overlap:     {len(overlap)}')
        if overlap:
            print(f'    Shared:       {sorted(overlap)}')
        if scannet_only:
            print(f'    ScanNet only: {sorted(scannet_only)}')
        if sun_only:
            print(f'    SUN only:     {sorted(sun_only)}')
        sun_to_scannet_pct = len(overlap) / len(sun_classes_lower) * 100 if sun_classes_lower else 0
        scannet_to_sun_pct  = len(overlap) / len(scannet_classes_lower) * 100 if scannet_classes_lower else 0
        print(f'  Coverage:  {sun_to_scannet_pct:.0f}% of SUN classes are in ScanNet  '
              f'(SUN -> ScanNet: transfer-learning relevance)')
        print(f'  Coverage:  {scannet_to_sun_pct:.0f}% of ScanNet classes are in SUN  '
              f'(ScanNet -> SUN: source coverage)')
        if sun_to_scannet_pct > 50:
            print(f'  Class overlap: GOOD')
        else:
            print(f'  Class overlap: LOW')
            print('  Note: Pretraining still helps even with low class overlap;')
            print('  the model learns general spatial features, not just class-specific ones.')

        # =====================================================================
        # 4. Dataset Scale Comparison
        # =====================================================================
        print()
        print('=' * 70)
        print('4. DATASET SCALE COMPARISON')
        print('=' * 70)
        scannet_train = sum(1 for _ in glob.glob(os.path.join(BASE_OUT_DIR, 'train', '*', '*_rgb.pt')))
        scannet_val   = sum(1 for _ in glob.glob(os.path.join(BASE_OUT_DIR, 'val',   '*', '*_rgb.pt')))
        sun_labels_path = os.path.join(sun_root, 'train', 'labels.txt')
        sun_train = sum(1 for line in open(sun_labels_path) if line.strip()) if os.path.exists(sun_labels_path) else 0

        print(f'  ScanNet train samples: {scannet_train:,}')
        print(f'  ScanNet val samples:   {scannet_val:,}')
        print(f'  ScanNet total:         {scannet_train + scannet_val:,}')
        print(f'  SUN RGB-D train:       {sun_train:,}')
        if sun_train > 0:
            ratio = (scannet_train + scannet_val) / sun_train
            print(f'  ScanNet/SUN ratio:     {ratio:.1f}x')
            if ratio >= 5:
                print(f'  Scale: EXCELLENT — ScanNet is {ratio:.0f}x larger')
            elif ratio >= 2:
                print(f'  Scale: GOOD — ScanNet is {ratio:.0f}x larger')
            else:
                print(f'  Scale: MARGINAL — pretraining data ideally 5-10x larger')

        # =====================================================================
        # 5. Summary Verdict
        # =====================================================================
        print()
        print('=' * 70)
        print('5. PRETRAINING FITNESS SUMMARY')
        print('=' * 70)
        issues = []
        if rgb_mean_diff > 0.10:
            issues.append(f'Large RGB mean divergence ({rgb_mean_diff:.3f})')
        if rgb_std_diff > 0.15:
            issues.append(f'Large RGB std divergence ({rgb_std_diff:.3f})')
        if HAS_SCANNET_HHA and SUN_HAS_HHA and hha_max_diff_rel > 0.50:
            issues.append(f'Large HHA-channel divergence ({hha_max_diff_rel:.1%})')

        if not issues:
            print(f'  VERDICT: ScanNet is well-suited for pretraining on {SUN_LABEL}')
            print('  - RGB distributions are compatible')
            if HAS_SCANNET_HHA and SUN_HAS_HHA:
                print('  - HHA distributions are compatible')
            print('  - Sufficient scale for transfer learning')
        else:
            print('  VERDICT: Review the following concerns:')
            for issue in issues:
                print(f'    - {issue}')
            print('  These may not be blockers — pretraining often helps even with')
            print('  some distribution shift, as the model learns general features.')

        # Clean up
        shutil.rmtree(SUN_TMP, ignore_errors=True)
    print(f'\n  Cleaned up {SUN_TMP}')

## 18. Spot Check

Visualize random samples from each class with matplotlib (RGB + depth
side by side).

In [ ]:
random.seed(42)

# Show RGB plus whichever modality(ies) we have.
# hha-only:  RGB + 3 HHA channels = 4 cols
# with-hha:  RGB + depth + 3 HHA channels = 5 cols
# none:      RGB + depth = 2 cols
HAS_DEPTH_FILES = (HHA_MODE != 'hha-only')
HAS_HHA_FILES   = (HHA_MODE != 'none')
n_cols = 1 + (1 if HAS_DEPTH_FILES else 0) + (3 if HAS_HHA_FILES else 0)

fig, axes = plt.subplots(
    len(SCANNET_SCENE_TYPES), n_cols,
    figsize=(4 * n_cols, 3 * len(SCANNET_SCENE_TYPES))
)

for i, cls in enumerate(SCANNET_SCENE_TYPES):
    cls_dir = os.path.join(BASE_OUT_DIR, 'train', cls)
    rgb_files = sorted(glob.glob(os.path.join(cls_dir, '*_rgb.pt')))

    if not rgb_files:
        for j in range(n_cols):
            axes[i, j].set_title(f'{cls} (no samples)' if j == 0 else '')
            axes[i, j].axis('off')
        continue

    chosen = random.choice(rgb_files)
    rgb = torch.load(chosen, weights_only=True).numpy().transpose(1, 2, 0)
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(f'{cls} - RGB')
    axes[i, 0].axis('off')

    col = 1
    if HAS_DEPTH_FILES:
        depth_path = chosen.replace('_rgb.pt', '_depth.pt')
        if os.path.exists(depth_path):
            depth = torch.load(depth_path, weights_only=True).numpy().squeeze()
            axes[i, col].imshow(depth, cmap='viridis')
            axes[i, col].set_title(f'{cls} - Depth')
        axes[i, col].axis('off')
        col += 1
    if HAS_HHA_FILES:
        hha_path = chosen.replace('_rgb.pt', '_hha.pt')
        if os.path.exists(hha_path):
            hha = torch.load(hha_path, weights_only=True).to(torch.float32).numpy()
            axes[i, col].imshow(hha[0], cmap='viridis')
            axes[i, col].set_title('disparity (1/m)')
            axes[i, col + 1].imshow(hha[1], cmap='RdBu_r')
            axes[i, col + 1].set_title('height (m)')
            axes[i, col + 2].imshow(hha[2], cmap='twilight', vmin=0, vmax=180)
            axes[i, col + 2].set_title('angle (deg)')
        for j in (col, col + 1, col + 2):
            axes[i, j].axis('off')

plt.tight_layout()
plt.show()

## 19. Package & Upload to Drive

## 18b. Validation Gate (HHA only)
Hard gate: visualization + dataset shape + range contract. Must pass before package + upload.


In [ ]:
# === Phase 3.6 Validation Gate (HHA only) ===
# Hard gate: must pass before tar+upload. The preprocess notebook is the only
# place that touches real ScanNet data, so any real-data validation lives here.
if HHA_MODE != 'none':
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm

    # 1. Visualize 6 diverse samples from train.
    print('\n=== Validation Gate (HHA) ===')
    sample_paths_for_viz = []
    for cls in sorted(os.listdir(os.path.join(BASE_OUT_DIR, 'train')))[:6]:
        cls_dir = os.path.join(BASE_OUT_DIR, 'train', cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in sorted(os.listdir(cls_dir)):
            if f.endswith('_hha.pt'):
                sample_paths_for_viz.append((cls, os.path.join(cls_dir, f)))
                break
    n_viz = min(6, len(sample_paths_for_viz))
    if n_viz:
        fig, axes = plt.subplots(n_viz, 4, figsize=(13, 3.0 * n_viz))
        if n_viz == 1:
            axes = axes[None, :]
        for row, (cls, hha_path) in enumerate(sample_paths_for_viz[:n_viz]):
            rgb_path = hha_path.replace('_hha.pt', '_rgb.pt')
            rgb = torch.load(rgb_path, weights_only=True).permute(1, 2, 0).numpy()
            hha = torch.load(hha_path, weights_only=True).to(torch.float32).numpy()
            axes[row, 0].imshow(rgb); axes[row, 0].set_title(cls); axes[row, 0].axis('off')
            axes[row, 1].imshow(hha[0], cmap='viridis', vmin=0, vmax=3.0)
            axes[row, 1].set_title('disparity (1/m)'); axes[row, 1].axis('off')
            axes[row, 2].imshow(hha[1], cmap='RdBu_r', vmin=-2, vmax=2)
            axes[row, 2].set_title('height (m)'); axes[row, 2].axis('off')
            norm = TwoSlopeNorm(vmin=0.0, vcenter=90.0, vmax=180.0)
            axes[row, 3].imshow(hha[2], cmap='RdBu_r', norm=norm)
            axes[row, 3].set_title('angle (deg) [0=ceil, 90=wall, 180=floor]'); axes[row, 3].axis('off')
        plt.tight_layout(); plt.show()

    # 2. Real-data dataset shape + range contract.
    # Hardened: 64 samples per split, both train AND val checked, RGB and HHA
    # both get value-range sanity, filename pairing asserted defensively.
    print('\n=== Dataset shape + range contract ===')
    if LOCAL_REPO_PATH not in sys.path:
        sys.path.insert(0, LOCAL_REPO_PATH)
    from src.data_utils.scannet_pretrain_dataset import (
        ScanNetPretrainDataset,
        _discover_samples,
        _load_class_names,
        _load_norm_stats,
    )
    cn = _load_class_names(BASE_OUT_DIR)
    ns = _load_norm_stats(BASE_OUT_DIR)

    N_CHECK_PER_SPLIT = 64
    # R5 #25: HHA channels are NOT Gaussian. The angle channel is bimodal
    # (peaks near 0 ceiling and 180 floor); height has a long tail (looking
    # up to high ceilings). After per-channel mean/std normalization, |q99|
    # can legitimately exceed 3.5 -- a tight threshold here causes false
    # validation failures. Loosen to 5.0 (RGB still fits comfortably; HHA
    # outliers caught by the percentile inspection in cell 49 instead).
    Q99_THRESHOLD = 5.0
    MIN_SCENES_PER_CLASS = {'train': 5, 'val': 1}  # train floor catches collapse; val=1 only catches truly-empty class dirs (canonical split puts 1 scene/class in val for rare classes like 'apartment')

    def _validate_split(split_name):
        samples = _discover_samples(
            os.path.join(BASE_OUT_DIR, split_name), cn, use_hha=True,
        )
        if not samples:
            raise RuntimeError(
                f'No (rgb, hha) pairs in {BASE_OUT_DIR}/{split_name}. '
                f'Cells 22 (extract) and/or 35 (recovery) did not produce paired data.'
            )
        # FIX #4: was mis-labeled "Class-stratified" -- _rng.sample is uniform
        # over the population. That's what we want here: the q99 gate measures
        # a global normalization property, not per-class distributions.
        # Uniform random (fixed seed) avoids the bias from samples[:N] which
        # would be ~entirely from the first alphabetical class (apartment)
        # since _discover_samples returns sorted output.
        import random as _random
        _rng = _random.Random(42)
        check_samples = _rng.sample(samples, min(N_CHECK_PER_SPLIT, len(samples)))
        # Defensive: every (rgb, hha) tuple's filenames must share the same stem.
        for rgb_p, hha_p, _label in check_samples:
            rgb_stem = os.path.basename(rgb_p).replace('_rgb.pt', '')
            hha_stem = os.path.basename(hha_p).replace('_hha.pt', '')
            assert rgb_stem == hha_stem, (
                f'Filename pairing mismatch in {split_name}: '
                f'rgb={rgb_p}  hha={hha_p}'
            )

        ds = ScanNetPretrainDataset(
            data_root=BASE_OUT_DIR, split=split_name,
            samples=check_samples,
            class_names=cn, norm_stats=ns,
            crop_size=224, normalize=True, use_hha=True,
            # Validation gate must isolate norm_stats correctness from
            # augmentation noise — otherwise q99 is noisy across runs.
            rgb_aug_prob=0.0, rgb_aug_mag=0.0,
            depth_aug_prob=0.0, depth_aug_mag=0.0,
        )
        n_check = min(N_CHECK_PER_SPLIT, len(ds))
        if n_check == 0:
            raise RuntimeError(f'{split_name} dataset empty after construction.')

        rgbs, hhas = [], []
        for i in range(n_check):
            rgb, hha, label = ds[i]
            assert rgb.shape == (3, 224, 224), f'{split_name}[{i}] bad RGB shape {rgb.shape}'
            assert hha.shape == (3, 224, 224), f'{split_name}[{i}] bad HHA shape {hha.shape}'
            assert not torch.isnan(rgb).any(), f'{split_name}[{i}] RGB has NaN'
            assert not torch.isnan(hha).any(), f'{split_name}[{i}] HHA has NaN'
            assert not torch.isinf(rgb).any(), f'{split_name}[{i}] RGB has Inf'
            assert not torch.isinf(hha).any(), f'{split_name}[{i}] HHA has Inf'
            assert 0 <= int(label) < len(cn), f'{split_name}[{i}] label {label} out of range'
            rgbs.append(rgb)
            hhas.append(hha)

        rgb_q99 = torch.stack(rgbs).abs().quantile(0.99).item()
        hha_q99 = torch.stack(hhas).abs().quantile(0.99).item()
        print(f'  [{split_name}] {n_check} samples: shape OK, no NaN/Inf, labels in range')
        print(f'  [{split_name}] post-norm |q99|: rgb={rgb_q99:.3f}  hha={hha_q99:.3f}')
        assert rgb_q99 < Q99_THRESHOLD, (
            f'[{split_name}] RGB out of normalized range: q99_abs={rgb_q99:.3f} >= {Q99_THRESHOLD}. '
            f'rgb_mean / rgb_std in norm_stats may be wrong.'
        )
        assert hha_q99 < Q99_THRESHOLD, (
            f'[{split_name}] HHA out of normalized range: q99_abs={hha_q99:.3f} >= {Q99_THRESHOLD}. '
            f'hha_mean / hha_std in norm_stats may be wrong.'
        )

    for split in ('train', 'val'):
        _validate_split(split)

    # #35: per-class scene-count floor — catch "everything in one class" disasters.
    print('\n=== Per-class scene-count floor ===')
    import re as _re
    for split in ('train', 'val'):
        split_dir = os.path.join(BASE_OUT_DIR, split)
        if not os.path.isdir(split_dir):
            continue
        floor = MIN_SCENES_PER_CLASS[split]
        for cls in cn:
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_dir):
                # Canonical ScanNet val split has 0 scenes for some rare
                # classes (closet, computer_cluster, gym). For pretraining
                # this is a property of the canonical split, not a fault.
                # Warn loudly but don't block.
                print(f'  WARN: [{split}] class {cls!r} missing from this split '
                      f'(canonical ScanNet split puts 0 scenes here for some rare classes).')
                continue
            scenes = set()
            for f in os.listdir(cls_dir):
                m = _re.match(r'(scene\d+_\d+)_f', f)
                if m:
                    scenes.add(m.group(1))
            if len(scenes) < floor:
                raise RuntimeError(
                    f'[{split}] class {cls!r} has only {len(scenes)} scenes '
                    f'(floor: {floor}). Per-class metrics will be noise.'
                )
        print(f'  {split}: all {len(cn)} classes have >= {floor} scenes')

    # #28: train and val class sets must agree (no class present in only one).
    _classes_in = {}
    for _sp in ('train', 'val'):
        _classes_in[_sp] = set()
        _split_dir = os.path.join(BASE_OUT_DIR, _sp)
        if not os.path.isdir(_split_dir):
            continue
        for _cls in cn:
            _cd = os.path.join(_split_dir, _cls)
            if os.path.isdir(_cd) and any(f.endswith('_rgb.pt') for f in os.listdir(_cd)):
                _classes_in[_sp].add(_cls)
    _train_only = _classes_in['train'] - _classes_in['val']
    _val_only   = _classes_in['val']   - _classes_in['train']
    if _train_only:
        # Canonical ScanNet split design — not a defect. Per-class val accuracy
        # for these classes will be undefined; overall val loss/accuracy is fine.
        print(f'  WARN: classes in train but not val (canonical split): '
              f'{sorted(_train_only)}. Per-class val metrics undefined for these; '
              f'overall val accuracy still meaningful.')
    if _val_only:
        # This IS a real disaster — model couldn't possibly classify these.
        # Canonical ScanNet split should never produce this; if it does, the
        # rebalance moved too aggressively or class mapping changed.
        raise RuntimeError(
            f'Classes present in val but missing from train: {sorted(_val_only)}. '
            f'Model never sees these classes during training. This indicates '
            f'a real preprocessing bug, not just canonical-split sparsity.'
        )
    print(f'  Class symmetry: train and val share {len(_classes_in["train"])} classes')

    print('\n=== VALIDATION GATE PASSED — proceed to package + upload ===')


In [ ]:
# Final canonical tarball -> Drive at HHA_TARBALL_NAME (downstream notebooks
# rsync this filename) AND at the snapshot name (crash recovery).
# Uses Drive REST API (synchronous, verified on Google's servers).

# Skip if dataset unchanged AND we restored from the canonical Drive copy
# (the Drive canonical is already bit-identical to our local data).
if ('CHANGES_MADE' in dir() and not CHANGES_MADE
        and 'RESTORE_SOURCE' in dir() and RESTORE_SOURCE == 'canonical'):
    print('SKIPPED: dataset unchanged from Drive canonical — no upload needed.')
    print(f'  Drive canonical: MyDrive/datasets/{HHA_TARBALL_NAME}')
else:
    local_tar = f'/content/{HHA_TARBALL_NAME}'

    ensure_pigz()

    print(f'Creating tar.gz with pigz ({MAX_WORKERS} threads)...')
    print(f'  Source: {BASE_OUT_DIR}')
    print(f'  Local:  {local_tar}')
    subprocess.run(
        ['tar', '-I', f'pigz -p {CPU_WORKERS}', '-cf', local_tar,
         # CRIT-R4 #3: KEEP *.done markers in the canonical tarball.
         # Cell 16's Pass 2 incremental detection scans these markers to
         # decide whether a re-run needs Pass 2 supplementing. Excluding
         # them silently disabled the entire Pass 2 incremental design
         # for any restore from canonical.
         '--exclude=*.failed',         # permanent-failure markers
         '--exclude=*.tmp',            # leftover from interrupted atomic writes
         '--exclude=progress.json',    # in-flight progress json
         '-C', os.path.dirname(BASE_OUT_DIR), os.path.basename(BASE_OUT_DIR)],
        check=True,
    )
    local_size = os.path.getsize(local_tar)
    print(f'Local archive: {local_size / 1024**3:.2f} GB')

    # Upload to canonical name (consumed by HPO/training/scout notebooks).
    canonical_id = upload_to_drive_sync(
        local_tar, 'datasets', HHA_TARBALL_NAME, return_file_id=True,
    )

    # Snapshot copy: server-side files.copy — no bytes re-uploaded.
    snapshot_name = f'{HHA_TARBALL_NAME.replace(".tar.gz", "")}.snapshot.tar.gz'
    copy_drive_file(canonical_id, 'datasets', snapshot_name)

    os.remove(local_tar)
    print('\nDone. Both files verifiably on Google servers.')
    print(f'  MyDrive/datasets/{HHA_TARBALL_NAME}  (canonical)')
    print(f'  MyDrive/datasets/{snapshot_name}  (snapshot)')

In [ ]:
# =====================================================================
# DISABLED: this cell was a one-shot recovery for a previously corrupted
# dataset. With cells 33 (DELETE_UNPAIRED=True), 35 (RUN_RGB_RECOVERY=True),
# 55 (verified canonical upload), and 24/37 (verified snapshot uploads)
# all working correctly, this cell is redundant on a fresh run-all.
# Leave commented unless you need to manually re-fix a damaged dataset.
# =====================================================================

# # =====================================================================
# # FINAL: fix /dev/shm dataset + upload verified copy to Drive at the
# # canonical name. After this prints "DRIVE READY", any ScanNet HHA
# # notebook (HPO, training, scout) can rsync + extract from Drive and
# # get a complete clean dataset. Idempotent — safe to re-run if it errors.
# # =====================================================================
#
# import os, re, sys, glob, shutil, subprocess, time
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import numpy as np
# import torch
# import cv2
# from tqdm.auto import tqdm
#
# DATA_ROOT     = '/dev/shm/scannet_pretrain_256_hha'
# DRIVE_DIR     = '/content/drive/MyDrive/datasets'
# CANONICAL_TAR = os.path.join(DRIVE_DIR, 'scannet_pretrain_256_hha.tar.gz')
# SNAPSHOT_TAR  = os.path.join(DRIVE_DIR, 'scannet_pretrain_256_hha.snapshot.tar.gz')
# TARGET_SIZE   = 256
# TOOLS_DIR     = '/content/scannet_recovery_tools'
# TMP_DIR       = '/dev/shm/_recovery_tmp'
# MAX_WORKERS   = min(8, os.cpu_count() or 4)
#
# assert os.path.isdir(DATA_ROOT), f'Dataset not found at {DATA_ROOT}'
# os.makedirs(TOOLS_DIR, exist_ok=True)
# os.makedirs(TMP_DIR, exist_ok=True)
# os.makedirs(DRIVE_DIR, exist_ok=True)
#
# # ---------- 1. Set up SensorData tools ----------
# dl_path = os.path.join(TOOLS_DIR, 'download-scannet.py')
# sd_path = os.path.join(TOOLS_DIR, 'SensorData.py')
# if not os.path.exists(dl_path):
#     subprocess.run(
#         ['wget', '-q', '-O', dl_path,
#          'http://kaldir.vc.cit.tum.de/scannet/download-scannet.py'],
#         check=True,
#     )
# if not os.path.exists(sd_path):
#     subprocess.run(
#         ['wget', '-q', '-O', sd_path,
#          'https://raw.githubusercontent.com/ScanNet/ScanNet/master/SensReader/python/SensorData.py'],
#         check=True,
#     )
#     with open(sd_path) as f:
#         lines = f.readlines()
#     for i, line in enumerate(lines):
#         m = re.match(r"^(\s*)print (.+)$", line.rstrip())
#         if m and 'print(' not in line:
#             lines[i] = f"{m.group(1)}print({m.group(2)})\n"
#         if ".join(struct.unpack(" in lines[i] and "b'" not in lines[i].split('join')[0]:
#             lines[i] = lines[i].replace("''.join(", "b''.join(")
#         if 'np.fromstring' in lines[i]:
#             lines[i] = lines[i].replace('np.fromstring', 'np.frombuffer')
#     with open(sd_path, 'w') as f:
#         f.writelines(lines)
# if TOOLS_DIR not in sys.path:
#     sys.path.insert(0, TOOLS_DIR)
# import SensorData
#
# # ---------- 2. Find zero-byte _rgb.pt; delete; map scene -> indices ----------
# print(f'Scanning {DATA_ROOT}...')
# scene_to_dir, scene_to_idx, n_zero = {}, {}, 0
# for split in ('train', 'val'):
#     sd_root = os.path.join(DATA_ROOT, split)
#     if not os.path.isdir(sd_root):
#         continue
#     for cls_entry in os.scandir(sd_root):
#         if not cls_entry.is_dir():
#             continue
#         for fname in os.listdir(cls_entry.path):
#             if not fname.endswith('_rgb.pt'):
#                 continue
#             fp = os.path.join(cls_entry.path, fname)
#             if os.path.getsize(fp) == 0:
#                 n_zero += 1
#                 m = re.match(r'(scene\d+_\d+)_f(\d+)_rgb\.pt$', fname)
#                 if m:
#                     sid, idx = m.group(1), int(m.group(2))
#                     scene_to_dir[sid] = cls_entry.path
#                     scene_to_idx.setdefault(sid, []).append(idx)
#                     os.remove(fp)
# print(f'Zero-byte _rgb.pt deleted: {n_zero}  across {len(scene_to_idx)} scenes')
#
# # ---------- 3. Per-scene recovery (multiprocessing) ----------
# def _recover_one(args):
#     sid, scene_dir, indices, tmp_dir, tools_dir, target_size = args
#     sens_dl = os.path.join(tmp_dir, sid)
#     os.makedirs(sens_dl, exist_ok=True)
#     sens_path = os.path.join(sens_dl, 'scans', sid, f'{sid}.sens')
#     written = 0
#     try:
#         if not os.path.exists(sens_path):
#             cmd = ['python3', os.path.join(tools_dir, 'download-scannet.py'),
#                    '-o', sens_dl, '--id', sid, '--type', '.sens']
#             subprocess.run(cmd, capture_output=True, text=True, timeout=600, input='y\ny\n')
#         if not os.path.exists(sens_path):
#             return (sid, 'download_failed', 0, len(indices))
#         sd_obj = SensorData.SensorData(sens_path)
#         depth_h, depth_w = sd_obj.depth_height, sd_obj.depth_width
#         for idx in indices:
#             if idx >= len(sd_obj.frames):
#                 continue
#             out_path = os.path.join(scene_dir, f'{sid}_f{idx:05d}_rgb.pt')
#             try:
#                 color_img = sd_obj.frames[idx].decompress_color(sd_obj.color_compression_type)
#                 if color_img is None:
#                     continue
#                 if color_img.shape[:2] != (depth_h, depth_w):
#                     color_img = cv2.resize(
#                         color_img, (depth_w, depth_h), interpolation=cv2.INTER_AREA,
#                     )
#                 color_crop = cv2.resize(
#                     color_img, (target_size, target_size), interpolation=cv2.INTER_AREA,
#                 )
#                 rgb_tensor = torch.from_numpy(
#                     color_crop.transpose(2, 0, 1).copy()
#                 ).to(torch.uint8)
#                 tmp_out = out_path + '.tmp'
#                 torch.save(rgb_tensor, tmp_out)
#                 fd = os.open(tmp_out, os.O_RDONLY)
#                 os.fsync(fd)
#                 os.close(fd)
#                 os.rename(tmp_out, out_path)
#                 if os.path.getsize(out_path) > 0:
#                     written += 1
#             except Exception:
#                 continue
#         return (sid, 'ok', written, len(indices))
#     finally:
#         shutil.rmtree(sens_dl, ignore_errors=True)
#
#
# if scene_to_idx:
#     tasks = [
#         (sid, scene_to_dir[sid], sorted(scene_to_idx[sid]),
#          TMP_DIR, TOOLS_DIR, TARGET_SIZE)
#         for sid in sorted(scene_to_idx)
#     ]
#     # Sanity-check first scene inline before launching pool — surfaces any
#     # worker-code error in <1 min instead of hanging tqdm at 0.
#     print(f'\nSanity check: recovering first scene inline...')
#     _t0 = time.time()
#     _r = _recover_one(tasks[0])
#     print(f'  {_r[0]}: status={_r[1]}, written={_r[2]}/{_r[3]}, took {time.time()-_t0:.1f}s')
#
#     if len(tasks) > 1:
#         # ThreadPoolExecutor (NOT multiprocessing.Pool): forking workers in
#         # Jupyter AFTER matplotlib/torch state has accumulated in the parent
#         # kernel can hang on worker startup. Threads share state cleanly.
#         # subprocess.run + torch.save release the GIL, so we still get
#         # parallelism on the network-bound + I/O-bound parts.
#         print(f'Recovering remaining {len(tasks)-1} scenes ({MAX_WORKERS} threads)...')
#         with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
#             futs = [ex.submit(_recover_one, t) for t in tasks[1:]]
#             for f in tqdm(as_completed(futs), total=len(futs), desc='Recovering'):
#                 try:
#                     f.result()
#                 except Exception as e:
#                     print(f'\n  Worker error (skipped): {type(e).__name__}: {e}')
#     shutil.rmtree(TMP_DIR, ignore_errors=True)
#
# # ---------- 4. Strip any remaining orphans/zero-byte (clean paired set) ----------
# print('\nFinal cleanup pass...')
# n_zero_left, n_orphan_rgb, n_orphan_hha = 0, 0, 0
# for split in ('train', 'val'):
#     sd_root = os.path.join(DATA_ROOT, split)
#     if not os.path.isdir(sd_root):
#         continue
#     for cls_entry in os.scandir(sd_root):
#         if not cls_entry.is_dir():
#             continue
#         for fname in list(os.listdir(cls_entry.path)):
#             fp = os.path.join(cls_entry.path, fname)
#             if fname.endswith('_rgb.pt'):
#                 if os.path.getsize(fp) == 0:
#                     n_zero_left += 1
#                     os.remove(fp)
#                     hha = fp.replace('_rgb.pt', '_hha.pt')
#                     if os.path.exists(hha):
#                         os.remove(hha)
#                         n_orphan_hha += 1
#                 elif not os.path.exists(fp.replace('_rgb.pt', '_hha.pt')):
#                     n_orphan_rgb += 1
#                     os.remove(fp)
#             elif fname.endswith('_hha.pt'):
#                 if not os.path.exists(fp.replace('_hha.pt', '_rgb.pt')):
#                     n_orphan_hha += 1
#                     os.remove(fp)
# print(f'  Removed still-zero _rgb.pt:   {n_zero_left}')
# print(f'  Removed orphan _rgb.pt:       {n_orphan_rgb}')
# print(f'  Removed orphan _hha.pt:       {n_orphan_hha}')
#
# # ---------- 5. Verify with the actual dataset class downstream notebooks use ----------
# print('\nVerifying via ScanNetPretrainDataset...')
# PROJECT_ROOT = '/content/Multi-Stream-Neural-Networks'
# if PROJECT_ROOT not in sys.path:
#     sys.path.insert(0, PROJECT_ROOT)
# from src.data_utils.scannet_pretrain_dataset import (
#     ScanNetPretrainDataset, _load_norm_stats, _load_class_names, _discover_samples,
# )
# ns = _load_norm_stats(DATA_ROOT)
# cn = _load_class_names(DATA_ROOT)
# train_samples = _discover_samples(os.path.join(DATA_ROOT, 'train'), cn, use_hha=True)
# val_samples   = _discover_samples(os.path.join(DATA_ROOT, 'val'),   cn, use_hha=True)
# print(f'  train samples: {len(train_samples)}')
# print(f'  val   samples: {len(val_samples)}')
# ds = ScanNetPretrainDataset(
#     data_root=DATA_ROOT, split='train',
#     samples=train_samples[:32], class_names=cn, norm_stats=ns,
#     crop_size=224, normalize=True, use_hha=True,
# )
# for i in range(min(10, len(ds))):
#     rgb, hha, _ = ds[i]
#     assert rgb.shape == (3, 224, 224) and hha.shape == (3, 224, 224)
#     assert not torch.isnan(rgb).any() and not torch.isnan(hha).any()
# print('  Dataset loads cleanly (10 samples, no NaN, correct shape).')
#
# # ---------- 6. Tar + upload to BOTH names with verified Drive sync ----------
# subprocess.run(
#     ['bash', '-c', 'which pigz >/dev/null 2>&1 || apt-get install -y -qq pigz'],
#     check=False, capture_output=True,
# )
# local_tar = '/dev/shm/_dataset_upload.tar.gz'
# print(f'\nCompressing {DATA_ROOT} -> {local_tar} (pigz, {MAX_WORKERS} threads)...')
# subprocess.run(
#     ['tar', '-I', f'pigz -p {MAX_WORKERS}', '-cf', local_tar,
#      '-C', os.path.dirname(DATA_ROOT), os.path.basename(DATA_ROOT)],
#     check=True,
# )
# local_size = os.path.getsize(local_tar)
# print(f'Local tarball: {local_size / 1024**3:.2f} GB')
#
# from google.colab import drive
# for dest in (CANONICAL_TAR, SNAPSHOT_TAR):
#     print(f'\nCopying to {dest} ...')
#     shutil.copy2(local_tar, dest)
#     print('  Forcing Drive sync (flush_and_unmount + remount)...')
#     drive.flush_and_unmount()
#     time.sleep(5)
#     drive.mount('/content/drive', force_remount=True)
#     drive_size = os.path.getsize(dest)
#     print(f'  Post-sync size on Drive: {drive_size / 1024**3:.2f} GB')
#     assert drive_size == local_size, (
#         f'UPLOAD FAILED for {dest}: local={local_size}, drive={drive_size}. '
#         f'Re-run this cell.'
#     )
#     print(f'  VERIFIED: {dest}')
#
# os.remove(local_tar)
# print('\n' + '=' * 70)
# print('  DRIVE READY')
# print(f'  Canonical: {CANONICAL_TAR}  ({drive_size / 1024**3:.2f} GB)')
# print(f'  Snapshot:  {SNAPSHOT_TAR}  ({drive_size / 1024**3:.2f} GB)')
# print(f'  Train: {len(train_samples)}  Val: {len(val_samples)}')
# print('  Any ScanNet HHA notebook can now use this dataset directly.')
# print('=' * 70)
#